# KEN Model — Causal Loop Diagrams

This notebook contains **causal loop diagrams** for every module of the KEN SFC model.
Each diagram is written as a **Mermaid code block** inside a markdown cell — VS Code renders them automatically (no need to "run" anything).

### How to modify a diagram
- Double-click any diagram's markdown cell to edit the Mermaid source.
- Press `Esc` or click outside to re-render.
- [Mermaid Flowchart Syntax Reference](https://mermaid.js.org/syntax/flowchart.html)

| Syntax | Meaning |
|--------|---------|
| `graph TB` | Top-to-bottom layout (`LR` = left-to-right) |
| `A --> B` | Arrow from A to B |
| `A -->\|label\| B` | Labeled arrow |
| `A["Text"]` | Node with text label |
| `subgraph "Name" ... end` | Group of nodes |

> **Tip:** If you also want a standalone HTML file with all diagrams, run the export cell at the bottom.

In [5]:
import json, re, math, base64, time
from pathlib import Path
from collections import defaultdict, deque

# ═══════════════════════════════════════════════════════════════════════
# Helper: extract all diagrams from the notebook
# ═══════════════════════════════════════════════════════════════════════
def _extract_diagrams(notebook_path="causal_loop_diagrams.ipynb"):
    """Return list of (title, mermaid_code, is_sub, parent_num) tuples.

    parent_num is the integer from the cell's ## N. heading (0 if absent).
    The extra 4th element is backward-compatible: callers that only unpack
    three values should use  ``for title, code, is_sub, *_ in diagrams``.
    """
    nb = json.loads(Path(notebook_path).read_text(encoding="utf-8"))
    diagrams = []
    for cell in nb["cells"]:
        if cell["cell_type"] != "markdown":
            continue
        src = "".join(cell["source"])
        mermaid_blocks = re.findall(r'```mermaid\s*\n(.*?)```', src, re.DOTALL)
        if not mermaid_blocks:
            continue
        main_heading = re.search(r'^##\s+(.+)', src, re.MULTILINE)
        main_title = main_heading.group(1).strip() if main_heading else None
        num_m = re.match(r'^(\d+)\.', main_title or "")
        parent_num = int(num_m.group(1)) if num_m else 0

        for i, block in enumerate(mermaid_blocks):
            sg = re.search(r'subgraph\s+"([^"]+)"', block)
            if i == 0 and main_title and len(mermaid_blocks) == 1:
                title = main_title
                is_sub = False
            elif sg:
                title = sg.group(1)
                is_sub = True
            elif main_title:
                title = main_title
                is_sub = False
            else:
                title = None
                is_sub = False
            diagrams.append((title, block, is_sub, parent_num))
    return diagrams

def _safe_filename(title, idx):
    """Convert a diagram title to a safe filename prefix."""
    if not title:
        return f"diagram_{idx:02d}"
    name = re.sub(r'[^a-zA-Z0-9]+', '_', title).strip('_').lower()
    return f"{idx:02d}_{name}"

# ═══════════════════════════════════════════════════════════════════════
# Export all diagrams to a single HTML file
# ═══════════════════════════════════════════════════════════════════════
def export_all_to_html(notebook_path="causal_loop_diagrams.ipynb",
                       output="Figures/causal_loop_diagrams.html"):
    """Export all diagrams to a single standalone HTML file."""
    diagrams = _extract_diagrams(notebook_path)
    parts = [
        '<!DOCTYPE html><html><head><meta charset="utf-8">',
        '<script src="https://cdn.jsdelivr.net/npm/mermaid/dist/mermaid.min.js"></script>',
        '<script>mermaid.initialize({startOnLoad:true, theme:"default", themeVariables:{fontSize:"18px"}, flowchart:{useMaxWidth:true,htmlLabels:true}});</script>',
        '<style>',
        '  body{font-family:Arial,sans-serif;max-width:1800px;margin:auto;padding:20px}',
        '  .diagram{margin:30px 0;border:1px solid #ddd;padding:20px;border-radius:8px;overflow-x:auto}',
        '  h2{color:#2c3e50}',
        '  .loop-title{font-size:1.4em;font-weight:bold;color:#2c3e50;margin:0 0 12px 0}',
        '  .mermaid svg{max-width:100%;height:auto;display:block}',
        '</style></head><body>',
        '<h1>KEN Model &mdash; Causal Loop Diagrams</h1>',
    ]
    for title, code, is_sub, *_ in diagrams:
        parts.append('<div class="diagram">')
        if title:
            if is_sub:
                parts.append(f'<p class="loop-title">{title}</p>')
            else:
                parts.append(f'<h2>{title}</h2>')
        parts.append(f'<div class="mermaid">\n{code.strip()}\n</div></div>')
    parts.append('</body></html>')
    out = Path(output)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text("\n".join(parts), encoding="utf-8")
    print(f"Exported {len(diagrams)} diagrams to {output}")

# ═══════════════════════════════════════════════════════════════════════
# Export diagrams to PNG/SVG image files (for LaTeX)
# Uses kroki.io POST API — works behind corporate proxies
# ═══════════════════════════════════════════════════════════════════════
def export_diagrams_to_images(notebook_path="causal_loop_diagrams.ipynb",
                              output_dir="Figures",
                              fmt="png",
                              theme="default"):
    """
    Export each mermaid diagram to an image file via the kroki.io API.

    Parameters
    ----------
    fmt : str   'png' or 'svg'. SVG is vector-quality (best for LaTeX).
    theme : str 'default', 'dark', 'forest', 'neutral'.

    LaTeX usage
    -----------
    PNG:  \\includegraphics[width=\\textwidth]{Figures/01_overall_model_architecture.png}
    SVG:  \\includesvg[width=\\textwidth]{Figures/01_overall_model_architecture.svg}
          (requires \\usepackage{svg} and Inkscape installed)
    """
    import requests
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

    diagrams = _extract_diagrams(notebook_path)
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)

    def _add_theme(code, theme):
        code = code.strip()
        if "%%{init:" not in code:
            return f'%%{{init: {{"theme": "{theme}", "themeVariables": {{"fontSize": "18px"}}}}}}%%\n{code}'
        return code

    scale = "?scale=2" if fmt == "png" else ""
    kroki_url = f"https://kroki.io/mermaid/{fmt}{scale}"
    saved = []

    for idx, (title, code, is_sub, *_) in enumerate(diagrams, 1):
        fname = _safe_filename(title, idx)
        fpath = out / f"{fname}.{fmt}"
        themed_code = _add_theme(code, theme)
        try:
            resp = requests.post(
                kroki_url,
                data=themed_code.encode("utf-8"),
                headers={"Content-Type": "text/plain"},
                timeout=60,
                verify=False,
            )
            if resp.status_code == 200:
                fpath.write_bytes(resp.content)
                saved.append(str(fpath))
                print(f"  [{idx:2d}/{len(diagrams)}] OK  {fpath}")
            else:
                print(f"  [{idx:2d}/{len(diagrams)}] FAIL ({resp.status_code}): {title}")
                print(f"       {resp.text[:200]}")
        except Exception as e:
            print(f"  [{idx:2d}/{len(diagrams)}] ERROR: {title} — {type(e).__name__}: {e}")
        time.sleep(0.5)

    print(f"\nSaved {len(saved)}/{len(diagrams)} images to {output_dir}/")
    if saved and diagrams:
        print(f"\nLaTeX usage:")
        if fmt == "png":
            print(f'  \\includegraphics[width=\\textwidth]{{Figures/{_safe_filename(diagrams[0][0], 1)}.{fmt}}}')
        else:
            print(f'  \\includesvg[width=\\textwidth]{{Figures/{_safe_filename(diagrams[0][0], 1)}.{fmt}}}')
    return saved

# ═══════════════════════════════════════════════════════════════════════
# Export nodes & edges for kumu.io
# ═══════════════════════════════════════════════════════════════════════
def export_for_kumu(notebook_path="causal_loop_diagrams.ipynb",
                    output_dir="Figures"):
    """
    Export diagram nodes and edges for kumu.io import.

    Produces in output_dir/ (ALL old kumu_*.* files are removed first):
      kumu_<slug>.json       – per-diagram blueprint, 1 per CLD (19 total)
      kumu_combined.json     – full overview (all 428 nodes)
      kumu_ken_model.xlsx    – Excel: Elements + Connections sheets
      kumu_view_settings.txt – paste into Kumu Advanced Editor

    Key fixes vs. earlier versions:
      • Flat fields (no nested "attributes") — Kumu's required format
      • tags is a JSON array  e.g. ["CLD 1", "CLD 4"]  (not pipe-string)
      • x / y positions pre-computed (hierarchical LR layout for causal
        chains; circular layout for pure feedback loops)
      • View settings use layout: static so positions are respected
        immediately — no waiting for force simulation to settle
    """

    # ── colour palette ─────────────────────────────────────────────────
    MODULE_COLORS = {
        "demand":            "#f39c12",
        "prices":            "#f39c12",
        "wages":             "#f39c12",
        "consumption":       "#f39c12",
        "investment":        "#2980b9",
        "export":            "#27ae60",
        "import":            "#27ae60",
        "green export":      "#27ae60",
        "balance of pay":    "#c0392b",
        "bop":               "#c0392b",
        "current account":   "#c0392b",
        "financial account": "#c0392b",
        "government":        "#8e44ad",
        "fiscal":            "#8e44ad",
        "sovereign":         "#d35400",
        "pif":               "#d35400",
        "nfa":               "#d35400",
        "two-phase":         "#d35400",
        "energy":            "#16a085",
        "fdi":               "#e74c3c",
        "repatriat":         "#e74c3c",
        "water":             "#3498db",
        "irrigation":        "#3498db",
        "io table":          "#7f8c8d",
        "supply":            "#7f8c8d",
        "stock-flow":        "#9b59b6",
        "household":         "#9b59b6",
        "multiplier":        "#f39c12",
        "profit":            "#2980b9",
        "wage-price":        "#e67e22",
        "oil":               "#d35400",
        "remittance":        "#e74c3c",
        "external debt":     "#c0392b",
        "feedback":          "#95a5a6",
    }

    def _get_color(group_name):
        g = group_name.lower()
        for key, color in MODULE_COLORS.items():
            if key in g:
                return color
        return "#bdc3c7"

    # ── layout: hierarchical LR for DAGs, circular for pure loops ──────
    def _compute_layout(elem_dict, conn_list):
        """
        Return {label: (x, y)} pixel positions for a single CLD map.

        Algorithm
        ---------
        1. Build directed graph from conn_list (edges within this diagram).
        2. Kahn BFS topological sort → assign column (x-level) to each node.
           Nodes are placed at column = max(parent column) + 1.
           Unvisited nodes (part of a feedback cycle) are identified.
        3. If ALL nodes are in cycles → full circular layout.
           Otherwise  → hierarchical for acyclic nodes,
                        separate circle cluster for cycle nodes.
        4. Within each column, nodes are sorted by subgraph type then label
           and stacked vertically.
        """
        labels    = list(elem_dict.keys())
        label_set = set(labels)
        n         = len(labels)
        if n == 0:
            return {}
        if n == 1:
            return {labels[0]: (0, 0)}

        adj_out = defaultdict(set)
        adj_in  = defaultdict(set)
        for c in conn_list:
            f, t = c["from"], c["to"]
            if f in label_set and t in label_set and f != t:
                adj_out[f].add(t)
                adj_in[t].add(f)

        # Kahn's topological BFS
        in_deg = {lbl: len(adj_in.get(lbl, set())) for lbl in labels}
        queue  = deque([lbl for lbl in labels if in_deg[lbl] == 0])
        level  = {lbl: 0 for lbl in queue}

        while queue:
            node = queue.popleft()
            for nbr in adj_out[node]:
                proposed = level[node] + 1
                level[nbr] = max(level.get(nbr, 0), proposed)
                in_deg[nbr] -= 1
                if in_deg[nbr] == 0:
                    queue.append(nbr)

        cycle_nodes = [lbl for lbl in labels if lbl not in level]

        # Pure circular layout (all in cycles — typical for feedback loops)
        if len(cycle_nodes) == n:
            r = max(220, 90 * math.log2(n + 1))
            return {
                lbl: (int(r * math.cos(2*math.pi*i/n - math.pi/2)),
                      int(r * math.sin(2*math.pi*i/n - math.pi/2)))
                for i, lbl in enumerate(labels)
            }

        # Cycle nodes → circular cluster placed to the right
        if cycle_nodes:
            max_col = max(level.values()) + 1
            r_cyc   = max(180, 80 * math.sqrt(len(cycle_nodes)))
            cx_cyc  = max_col * 450
            for i, lbl in enumerate(cycle_nodes):
                angle = 2*math.pi*i/len(cycle_nodes) - math.pi/2
                level[lbl] = max_col   # so they sort into their own column
                # Override with circular coords later
            cycle_pos = {
                lbl: (int(cx_cyc + r_cyc * math.cos(2*math.pi*i/len(cycle_nodes) - math.pi/2)),
                      int(r_cyc * math.sin(2*math.pi*i/len(cycle_nodes) - math.pi/2)))
                for i, lbl in enumerate(cycle_nodes)
            }
        else:
            cycle_pos = {}

        # Group acyclic nodes by level; sort within level by subgraph then label
        lev_groups = defaultdict(list)
        for lbl in labels:
            if lbl not in cycle_pos:
                lev_groups[level[lbl]].append(lbl)
        for col in lev_groups:
            lev_groups[col].sort(
                key=lambda l: (elem_dict.get(l, {}).get("type", ""), l)
            )

        H = 450   # horizontal spacing between columns
        V = 200   # vertical spacing between rows
        positions = {}
        for col, nodes in sorted(lev_groups.items()):
            x      = col * H
            y_top  = -((len(nodes) - 1) * V) // 2
            for i, lbl in enumerate(nodes):
                positions[lbl] = (x, y_top + i * V)

        positions.update(cycle_pos)
        return positions

    # ── view settings (static layout — positions used as-is) ───────────
    def _build_view_settings():
        seen, color_rules = set(), []
        for key, color in MODULE_COLORS.items():
            rule = 'element["type"*="{k}"] {{ color: {c}; }}'.format(k=key, c=color)
            if rule not in seen:
                seen.add(rule)
                color_rules.append(rule)
        cld_filters = [
            '/* element:not(["tags" contains "CLD {i}"]) {{ display: none; }} */'.format(i=i)
            for i in range(1, 14)
        ]
        lines = [
            "/* ═══════════════════════════════════════════════════════════════════ */",
            "/* KEN Model — Kumu View Settings  (paste into Advanced Editor / <>)  */",
            "/* ─────────────────────────────────────────────────────────────────── */",
            "/* Steps:                                                               */",
            "/*  1. Open your Kumu map                                               */",
            "/*  2. Click the  ⚙  Settings icon (right side)                        */",
            "/*  3. Click 'Customize view defaults'                                  */",
            "/*  4. Click '<>'  (Advanced Editor) in the top-right of the panel     */",
            "/*  5. Paste this entire block, replacing what's there                  */",
            "/*  6. Click Done → Save current view                                   */",
            "/* ═══════════════════════════════════════════════════════════════════ */",
            "",
            "/* layout: static uses the pre-computed x/y positions in the JSON.    */",
            "/* Switch to  layout: force  if you want physics-based re-layout.      */",
            "@settings {",
            "  layout: static;",
            "  font-size: 14;",
            "  element-color: categorize('type', neon2);",
            "}",
            "",
            "/* ── Fallback force-layout settings (enable by switching layout above) */",
            "/* @settings {                                                            */",
            "/*   layout: force;                                                       */",
            "/*   layout-gravity: 0;                                                   */",
            "/*   layout-particle-charge: 2000;                                        */",
            "/*   connection-length: 700;                                              */",
            "/*   connection-strength: 0.05;                                           */",
            "/* }                                                                      */",
            "",
            "/* ── Connection colors by polarity ── */",
            'connection["type"="positive"]    { color: #27ae60; stroke-width: 2; }',
            'connection["type"="negative"]    { color: #c0392b; stroke-width: 2; }',
            'connection["type"="unspecified"] { color: #95a5a6; }',
            "",
            "/* ── Element colors by module type ── */",
        ] + color_rules + [
            "",
            "/* ── Optional: isolate one CLD at a time ─────────────────────────── */",
            "/* Uncomment exactly one line to show only that CLD's elements:       */",
        ] + cld_filters
        return "\n".join(lines)

    # ── Mermaid parsing regexes ─────────────────────────────────────────
    edge_re     = re.compile(
        r'(\w+)\s*(?:\["([^"]+)"\])?\s*'
        r'-->\s*(?:\|([^|]*)\|\s*)?'
        r'(\w+)\s*(?:\["([^"]+)"\])?'
    )
    node_def_re = re.compile(r'(\w+)\s*\["([^"]+)"\]')
    subgraph_re = re.compile(r'subgraph\s+"([^"]+)"')

    def _diag_num(title):
        m = re.match(r'^(\d+)\.', title or "")
        return int(m.group(1)) if m else 0

    def _short_title(title):
        return re.sub(r'^\d+\.\s*', '', title or "").strip()

    # ── remove ALL stale kumu_* files first ────────────────────────────
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    stale = list(out.glob("kumu_*"))
    for f in stale:
        f.unlink()
    if stale:
        print(f"Removed {len(stale)} stale kumu_* file(s) from {output_dir}/")

    # ── parse diagrams ─────────────────────────────────────────────────
    diagrams        = _extract_diagrams(notebook_path)
    diag_data       = []
    all_elements    = {}   # label -> {type, color, diag_tags: set()}
    all_connections = {}   # (from, to) -> conn_dict

    _last_main = 0
    _sub_ctr   = 0

    for diag_idx, (diag_title, code, is_sub, parent_num) in enumerate(diagrams, 1):
        if not is_sub:
            diag_num    = _diag_num(diag_title) or diag_idx
            _last_main  = diag_num
            _sub_ctr    = 0
            cld_tag     = "CLD {n}".format(n=diag_num)
            short_title = _short_title(diag_title)
            slug        = _safe_filename(short_title, diag_num)
        else:
            _sub_ctr  += 1
            sub_label  = chr(ord('a') + _sub_ctr - 1)
            base_num   = parent_num or _last_main
            cld_tag    = "CLD {n}{l}".format(n=base_num, l=sub_label)
            short_title = diag_title or "sub_{n}".format(n=diag_idx)
            slug_raw   = _safe_filename(short_title, diag_idx)
            slug       = "{n:02d}{l}_{rest}".format(
                n=base_num, l=sub_label,
                rest=re.sub(r'^\d+_', '', slug_raw),
            )

        diag_elements    = {}
        diag_connections = []

        lines       = code.strip().splitlines()
        node_map    = {}
        group_stack = [short_title]
        cur_group   = short_title

        # Pass 1 — collect ID["Label"] definitions
        for line in lines:
            for m in node_def_re.finditer(line):
                node_map[m.group(1)] = m.group(2)

        # Pass 2 — elements & connections
        for line in lines:
            s = line.strip()
            if re.match(r'^graph\s+', s):
                continue
            sg = subgraph_re.match(s)
            if sg:
                cur_group = sg.group(1)
                group_stack.append(cur_group)
                continue
            if s == "end":
                if len(group_stack) > 1:
                    group_stack.pop()
                cur_group = group_stack[-1]
                continue

            em = edge_re.search(s)
            if em:
                fid, flabel, elabel, tid, tlabel = em.groups()
                if flabel:
                    node_map[fid] = flabel
                if tlabel:
                    node_map[tid] = tlabel
                fl    = node_map.get(fid, fid)
                tl    = node_map.get(tid, tid)
                color = _get_color(cur_group)
                diag_elements.setdefault(fl, {"type": cur_group, "color": color})
                diag_elements.setdefault(tl, {"type": cur_group, "color": color})

                elabel_clean = (elabel or "").strip().strip('"').strip()
                polarity = (
                    "positive"    if elabel_clean.startswith("+") else
                    "negative"    if elabel_clean.startswith("-") else
                    "unspecified"
                )
                diag_connections.append({
                    "from":      fl,
                    "to":        tl,
                    "label":     elabel_clean,
                    "type":      polarity,
                    "direction": "directed",
                })
            else:
                for m in node_def_re.finditer(s):
                    lbl = m.group(2)
                    diag_elements.setdefault(lbl, {
                        "type": cur_group, "color": _get_color(cur_group)
                    })

        # Compute layout positions for this diagram
        positions = _compute_layout(diag_elements, diag_connections)
        for lbl, (px, py) in positions.items():
            if lbl in diag_elements:
                diag_elements[lbl]["x"] = px
                diag_elements[lbl]["y"] = py

        # Accumulate into global sets
        for lbl, meta in diag_elements.items():
            if lbl not in all_elements:
                all_elements[lbl] = {
                    "type":      meta["type"],
                    "color":     meta["color"],
                    "diag_tags": set(),
                    "x":         meta.get("x", 0),
                    "y":         meta.get("y", 0),
                }
            all_elements[lbl]["diag_tags"].add(cld_tag)

        for c in diag_connections:
            all_connections.setdefault((c["from"], c["to"]), c)

        diag_data.append({
            "cld_tag":     cld_tag,
            "slug":        slug,
            "elements":    diag_elements,
            "connections": diag_connections,
        })

    # ── build Kumu element / connection objects ─────────────────────────
    def _elem_obj(label, type_str, color, diag_tags, x=0, y=0):
        # tags MUST be a JSON array for Kumu's blueprint format
        tags_list = sorted(diag_tags)
        desc = "Appears in: " + ", ".join(tags_list) if tags_list else ""
        return {
            "label":       label,
            "type":        type_str,
            "description": desc,
            "tags":        tags_list,   # ← JSON array, NOT pipe-separated string
            "x":           x,
            "y":           y,
        }

    def _conn_obj(c):
        return {
            "from":      c["from"],
            "to":        c["to"],
            "label":     c["label"],
            "type":      c["type"],
            "direction": "directed",
        }

    written = []
    print("Writing per-diagram JSON files:")

    # ── per-diagram JSON files ──────────────────────────────────────────
    for d in diag_data:
        elems = [
            _elem_obj(lbl, m["type"], m["color"], {d["cld_tag"]},
                      m.get("x", 0), m.get("y", 0))
            for lbl, m in d["elements"].items()
        ]
        conns = [_conn_obj(c) for c in d["connections"]]
        bp    = {"elements": elems, "connections": conns}
        fp    = out / "kumu_{s}.json".format(s=d["slug"])
        fp.write_text(json.dumps(bp, indent=2, ensure_ascii=False), encoding="utf-8")
        written.append(str(fp))
        print("  {tag}  ({ne} nodes, {nc} edges)  ->  {fn}".format(
            tag=d["cld_tag"], ne=len(elems), nc=len(conns), fn=fp.name))

    # ── combined JSON ───────────────────────────────────────────────────
    # Re-compute global layout so positions don't collide across diagrams
    all_pos = {}
    GRID_W, GRID_H = 3000, 2500   # spacing between diagram clusters
    cols = 4
    for gi, d in enumerate(diag_data):
        off_x = (gi % cols) * GRID_W
        off_y = (gi // cols) * GRID_H
        for lbl, (px, py) in _compute_layout(d["elements"], d["connections"]).items():
            all_pos[lbl] = (off_x + px, off_y + py)

    all_elem_list = [
        _elem_obj(lbl, m["type"], m["color"], m["diag_tags"],
                  all_pos.get(lbl, (0, 0))[0],
                  all_pos.get(lbl, (0, 0))[1])
        for lbl, m in all_elements.items()
    ]
    all_conn_list = [_conn_obj(c) for c in all_connections.values()]
    cp = out / "kumu_combined.json"
    cp.write_text(
        json.dumps({"elements": all_elem_list, "connections": all_conn_list},
                   indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    written.append(str(cp))
    print("  Overview  ({ne} nodes, {nc} edges)  ->  {fn}".format(
        ne=len(all_elem_list), nc=len(all_conn_list), fn=cp.name))

    # ── XLSX ────────────────────────────────────────────────────────────
    try:
        import openpyxl
        wb   = openpyxl.Workbook()
        ws_e = wb.active
        ws_e.title = "Elements"
        ws_e.append(["Label", "Type", "Description", "Tags", "X", "Y"])
        for lbl, m in all_elements.items():
            tags_str = "|".join(sorted(m["diag_tags"]))
            desc     = "Appears in: " + ", ".join(sorted(m["diag_tags"]))
            x, y     = all_pos.get(lbl, (0, 0))
            ws_e.append([lbl, m["type"], desc, tags_str, x, y])
        ws_c = wb.create_sheet("Connections")
        ws_c.append(["From", "To", "Label", "Type", "Direction"])
        for c in all_connections.values():
            ws_c.append([c["from"], c["to"], c["label"], c["type"], "directed"])
        xp = out / "kumu_ken_model.xlsx"
        wb.save(str(xp))
        written.append(str(xp))
        print("  XLSX  ({ne} elements, {nc} connections)  ->  {fn}".format(
            ne=len(all_elements), nc=len(all_connections), fn=xp.name))
    except ImportError:
        print("  WARNING: openpyxl not installed — run: pip install openpyxl")

    # ── view settings ───────────────────────────────────────────────────
    vp = out / "kumu_view_settings.txt"
    vp.write_text(_build_view_settings(), encoding="utf-8")
    written.append(str(vp))
    print("  View settings  ->  {fn}".format(fn=vp.name))

    print()
    print("Summary:")
    print("  Unique elements:    {n}".format(n=len(all_elements)))
    print("  Unique connections: {n}".format(n=len(all_connections)))
    print("  Diagrams exported:  {n}".format(n=len(diag_data)))
    print()
    print("KUMU IMPORT WORKFLOW")
    print("  1. Create a NEW Kumu project  →  kumu.io/new")
    print("  2. Per-diagram maps: for each kumu_XX_*.json file,")
    print("     add a new map in Kumu and drag-and-drop that file")
    print("     onto the blank canvas.  Click 'Save changes'.")
    print("  3. Overview map: drag-and-drop kumu_combined.json.")
    print("  4. Apply view settings: open the map, click ⚙ Settings,")
    print("     'Customize view defaults', '<>' Advanced Editor,")
    print("     paste kumu_view_settings.txt.  Click Done + Save.")
    print("  NOTE: the view sets layout:static — nodes stay at")
    print("        pre-computed positions.  Switch to layout:force")
    print("        in the settings if you want floating physics.")
    return written

print("Export helpers loaded.")
print("  export_all_to_html()        -> single HTML file")
print("  export_diagrams_to_images() -> PNG/SVG for LaTeX")
print("  export_for_kumu()           -> JSON/XLSX for kumu.io  (positions pre-set)")


Export helpers loaded.
  export_all_to_html()        -> single HTML file
  export_diagrams_to_images() -> PNG/SVG for LaTeX
  export_for_kumu()           -> JSON/XLSX for kumu.io  (positions pre-set)


## 1. Overall Model Architecture
Top-level view of the entire KEN simulation loop (model.py).

```mermaid
graph TB
    subgraph "Demand Side (Section 3-5)"
        P_prices["Prices p_(t)"] -->|"+ cost push"| INF["Inflation"]
        INF -->|"+ indexation"| W["Wages W_(t)"]
        W -->|"+ MPC wage"| C["Consumption C(t)"]
        PROFITS_lag["Profits P(t-1)"] -->|"+ MPC profit"| C
        WEALTH_lag["HH Wealth V(t-1)"] -->|"+ MPC wealth"| C
        INF -->|"+ nominal adj"| C
        REMIT["Remittances"] -->|"-  income drain"| C
    end

    subgraph "Investment Module (Section 4)"
        I["Investment I_(t)"]
        I_energy["Energy Investment"]
        I_desal["Desal/Wwater Investment"]
        I_FDI["FDI Net Flows"]
        I_energy -->|"+"| I
        I_desal -->|"+"| I
        I -->|"+ base for FDI"| I_FDI
        I_FDI -->|"-  domestic share"| I_DOM["Domestic Investment"]
    end

    subgraph "Exports Module (Section 5)"
        EX_oil["Oil Exports"]
        EX_nonoil["Non-Oil Exports"]
        EX_green["Green Exports"]
        EX_oil -->|"+"| EX["Total Exports"]
        EX_nonoil -->|"+"| EX
        EX_green -->|"+ adds"| EX_nonoil
    end

    subgraph "Government (Section 5.2)"
        GY["Gov Consumption GY(t)"]
        GOV_REV["Gov Revenue"]
        GOV_EXP["Gov Expenditure"]
        BOND["Gov Dom Bonds"]
        NFA_GOV["NFA_gov (Reserves)"]
        PIF_NODE["PIF (Sovereign Wealth)"]
        GOV_REV -->|"-  surplus reduces"| BOND
        GOV_EXP -->|"+ deficit raises"| BOND
        GOV_REV -->|"+ surplus to PIF"| PIF_NODE
        NFA_GOV -->|"+ returns extracted"| GOV_REV
        PIF_NODE -->|"+ accumulates"| NFA_GOV
    end

    subgraph "Supply Side (Section 6-7)"
        Qs["Supply Choice Q_s"]
        Qd["Aggregate Demand Q_d"]
        LEONTIEF["Leontief Production fn"]
        Ys["Realized Supply Y_s"]
        Qs -->|"+"| LEONTIEF
        Qd -->|"+"| Ys
        LEONTIEF -->|"+ constrained"| Ys
    end

    subgraph "IO Structure (Section 10)"
        A_MATRIX["A Matrix (IO Table)"]
        X["Total Output X_(t)"]
        V2030["Vision 2030 Policy"]
    end

    subgraph "Energy Module (Section 11)"
        ENERGY["Energy Demand"]
        EMISSIONS["Emissions"]
        RE["Renewables"]
        NETZERO["Net Zero Offsets"]
    end

    subgraph "Accounting (Section 12)"
        K["Capital Stock K"]
        PROFITS["Profits P_(t)"]
        HH["HH Accounting"]
        BANKS["Bank Profits"]
    end

    subgraph "Biophysical (Section 13)"
        WATER["Water Module"]
    end

    C -->|"+"| Qd
    GY -->|"+"| Qd
    I -->|"+"| Qd
    EX -->|"+"| Qd
    IM["Imports IM"] -->|"-"| Qd

    Ys -->|"+ Leontief"| X
    A_MATRIX -->|"+ matrix mult"| X
    V2030 -->|"+ growth targets"| A_MATRIX
    X -->|"-  unit cost share"| P_prices
    PROFITS -->|"+ markup"| P_prices

    X -->|"+ intensity"| ENERGY
    ENERGY -->|"+ combustion"| EMISSIONS
    RE -->|"-  displaces fossil"| EMISSIONS
    NETZERO -->|"-  removes"| EMISSIONS
    EMISSIONS -->|"-  carbon cost"| PROFITS

    I -->|"+"| K
    K -->|"+ capacity"| LEONTIEF
    W -->|"+ labour"| LEONTIEF

    X -->|"+ gross output"| PROFITS
    W -->|"-  wage cost"| PROFITS
    PROFITS -->|"+"| HH
    W -->|"+"| HH
    HH -->|"+"| WEALTH_lag
    I -->|"+ loans"| BANKS
    BANKS -->|"+ returns"| HH

    Ys -->|"+ gap fills"| IM
    X -->|"+ demand"| WATER

    WATER -->|"+ signal"| I_desal```

    REMIT -->|"-  outflow"| NFA_GOV

    GOV_REV -->|"+"| GOV_EXP    IM -->|"-  trade deficit"| NFA_GOV

    PROFITS -->|"+ tax base"| GOV_REV    EX -->|"+ trade surplus"| NFA_GOV

    C -->|"+ VAT"| GOV_REV    GY -->|"+"| GOV_EXP

## 2. Prices, Wages & Consumption
model.py §3.1–3.4: price formation (EMA normal-cost pricing), inflation, wage determination, household consumption.

```mermaid
graph LR
    subgraph "Price Formation & Inflation"
        X_lag["X_(t-1) Total Output"] -->|"-  unit cost share"| UCOST["Unit Cost EMA unit_cost_smooth_"]
        P_lag["Profits P_(t-1)"] -->|"-  reduces cost share"| UCOST
        UCOST -->|"+ normal-cost markup theta"| P_price["Prices p_(t)"]
        INF_E["Expected Inflation"] -->|"+ cost push"| P_price
        P_price -->|"+ Laspeyres index"| INF["GDP Deflator"]
        P_price -->|"+ consumer weights dC"| INF_C["Consumer Price Index"]
        INF -->|"+ persistence"| INF_E
        P_price -->|"fixed at 1"| DESAL_P["Desal/Wwater Price = 1"]
    end

    subgraph "Wage Determination"
        INF_lag["Inflation(t-1)"] -->|"+ indexation"| gW["Wage growth gW"]
        POP["Population growth adj."] -->|"+ additive"| gW
        gW -->|"+"| W["Wages W_(t)"]
        X_lag2["X_(t-1)"] -->|"+ wage share"| W
        SAUD["Saudization rate"] -->|"-  reduces foreign wage bill"| W
    end

    subgraph "Household Consumption"
        YD_W["Disposable Wage Income YD_wage"] -->|"+ MPC alpha1"| C["Consumption C(t)"]
        YD_P["Disposable Profit Income YD_profit"] -->|"+ MPC alpha2"| C
        V_lag["HH Wealth V(t-1)"] -->|"+ MPC alpha3"| C
        INF_E2["Inflation Forecast"] -->|"+ nominal adj"| C
        TAU_VAT["VAT rate tau_vat"] -->|"-  reduces net"| C
        REMIT["Remittances(t)"] -->|"-  reduces YD_wage"| YD_W
        C -->|"+ composition dC"| C_sect["Sectoral Consumption C_(t)"]
    end


    W -->|"+"| YD_W```

    TAX_W["Wage Tax"] -->|"-"| YD_W    TH["HH Taxes TH"] -->|"-"| YD_P

    P_lag -->|"+"| YD_P    FDI_INC["FDI Income Payments"] -->|"-  repatriation"| YD_P
    PB_HH["Bank Profits to HH"] -->|"+"| YD_P

## 3. Investment Module
investment_module.py + energy_investment_module.py + FDI_module.py

```mermaid
graph TB
    subgraph "Energy Investment Module"
        ELEC_LOAD["Electrification Added Load"] -->|"+ demand"| RE_TARGET["RE Generation Target"]
        V2030_RE["Vision 2030 RE Ramp"] -->|"+ policy"| RE_TARGET
        NZ_RE["Net Zero 85pct Target"] -->|"+ policy"| RE_TARGET
        RE_TARGET -->|"+ gap"| NEW_CAP["New RE Capacity Needed"]
        K_RE_lag["RE Capital(t-1)"] -->|"-  depreciated existing"| NEW_CAP
        NEW_CAP -->|"+ solar share"| I_SOLAR["I_solar"]
        NEW_CAP -->|"+ wind share"| I_WIND["I_wind"]
        NEW_CAP -->|"+ bio share"| I_BIO["I_bio"]
        I_SOLAR -->|"+"| I_RE["Total RE Investment I_RE"]
        I_WIND -->|"+"| I_RE
        I_BIO -->|"+"| I_RE
    end

    subgraph "Hydrogen Investment"
        P_H2["H2 capacity growth"] -->|"+"| I_H2["I_hydrogen"]
        K_H2["H2 capital stock"] -->|"+ capacity"| H2_PROD["H2 Production"]
    end

    subgraph "Net Zero Investment"
        CCS["CCS capacity ramp"] -->|"+"| I_CCS["I_netzero_ccs"]
        NAT["Nature-based SGI"] -->|"+"| I_NAT["I_netzero_nature"]
        CCU["CCU growth"] -->|"+"| I_CCU["I_netzero_ccu"]
    end

    subgraph "Efficiency & Electrification"
        EFF_GAIN["Efficiency annual gains"] -->|"+"| I_EFF["I_efficiency_total"]
        FOSSIL["Fossil demand"] -->|"+ annual rate shift"| I_ELECTR["I_electrification_total"]
        I_ELECTR -->|"+ adds load"| ELEC_LOAD
    end

    subgraph "Storage"
        RE_VAR["RE variability"] -->|"+ need"| I_STOR["I_storage"]
        LOCAL_SHARE["Local content ramp"] -->|"+"| I_STOR_LOCAL["I_storage_local"]
        I_STOR -->|"+"| I_STOR_LOCAL
    end

    subgraph "Desalination & Wastewater"
        WATER_GAP["Water gap"] -->|"+"| I_DESAL["I_desal"]
        V2030_WATER["Vision 2030 high growth"] -->|"+"| I_DESAL
        I_DESAL -->|"+"| K_DESAL["Desal Capital"]
        WATER_GAP -->|"+"| I_WWATER["I_wastewater"]
    end

    subgraph "General Sectoral Investment"
        I_TREND["gI_trend_endog"] -->|"+"| I_SECT["Sectoral I_(t)"]
        Y_DIST["Y_(t-1) sectoral shares"] -->|"+"| I_SECT
        EXCESS_IMP["Excess imports"] -->|"+ scarcity signal"| I_SECT
        K_GAP["Capital scarcity"] -->|"+ scarcity signal"| I_SECT
        I_RE -->|"+"| I_SECT
        I_H2 -->|"+"| I_SECT
        I_CCS -->|"+"| I_SECT
        I_NAT -->|"+"| I_SECT
        I_CCU -->|"+"| I_SECT
        I_EFF -->|"+"| I_SECT
        I_ELECTR -->|"+"| I_SECT
        I_STOR -->|"+"| I_SECT
    end

    subgraph "FDI Decomposition"
        I_SECT -->|"+ base"| FDI["FDI net flows"]
        I_SECT -->|"+ total"| I_DOM["Domestic Investment I_dom"]
        FDI -->|"-  substitutes"| I_DOM
        I_DOM -->|"+ private share"| I_PRIV_DOM["I_private_dom"]
        I_DOM -->|"+ public share"| I_PUB_DOM["I_public_dom"]
    end

    subgraph "Demand Vectors"
        I_SECT -->|"+ dI, dI_RE, dI_Effic vectors"| I_DEMAND["I_demand_(t)"]
        GRID["Grid upgrade demand"] -->|"+"| I_DEMAND
    end
```

## 4. Exports, Imports, Green Exports & Balance of Payments
exports_module.py + green_exports_module.py + FDI_module.py BoP closure.

```mermaid
graph TB
    subgraph "Oil Exports"
        OIL_TS["Oil Activity Time Series"] -->|"+ AR model"| OIL_FORECAST["Oil Growth Forecast"]
        OIL_ADJ["Oil growth adjustment param"] -->|"+"| OIL_G["Effective Oil Growth"]
        OIL_FORECAST -->|"+"| OIL_G
        SHOCK["Random Oil Shock sigma"] -->|"+/- stochastic"| OIL_G
        OIL_G -->|"+"| EX_OIL["Oil Exports EX_oil_(t)"]
        EX_OIL_lag["EX_oil_(t-1)"] -->|"+"| EX_OIL
        X_lag["X_(t-1)"] -->|"-  supply ceiling"| EX_OIL
    end

    subgraph "Non-Oil Exports"
        NONOIL_TS["Non-Oil Export Time Series"] -->|"+ AR model"| NONOIL_FORECAST["Non-Oil Growth Forecast"]
        NONOIL_ADJ["Non-oil export growth adj."] -->|"+"| NONOIL_G["Effective Non-Oil Growth"]
        V2030_EX["V2030 sectoral growth rates"] -->|"+ scaling factor"| NONOIL_G
        NONOIL_FORECAST -->|"+"| NONOIL_G
        NONOIL_G -->|"+"| EX_NONOIL["Non-Oil Exports EX_non_oil_(t)"]
        EX_NONOIL_lag["EX_non_oil_(t-1)"] -->|"+"| EX_NONOIL
    end

    subgraph "Green Exports  -  green_exports_module.py"
        H2_EXPORT["H2 Export Volume"] -->|"+ price per GJ"| H2_EX_VAL["H2 Export Value"]
        H2_EX_VAL -->|"+"| EX_NONOIL
        SOLAR_INV["Solar/Battery Investment"] -->|"+ local content pct"| SOLAR_EX["Solar Export Value"]
        SOLAR_EX -->|"+"| EX_NONOIL
        SOLAR_INV -->|"-  import substitution"| IM_REDUCE["Reduced Imports IM_(t)"]
        WIND_INV["Wind/Electrification Inv"] -->|"+ local content pct"| WIND_EX["Wind/Mach Export"]
        WIND_EX -->|"+"| EX_NONOIL
        WIND_INV -->|"-  import substitution"| IM_REDUCE
        CARBON_CREDITS["Carbon Credit Export"] -->|"+"| EX_NONOIL
        STORAGE_LOCAL["Local Storage Production"] -->|"-"| IM_REDUCE
    end

    subgraph "Imports"
        IM_lag["IM_(t-1)"] -->|"+"| IM["Imports IM_(t)"]
        IM_ADJ["Import growth adj."] -->|"+"| IM
        UNMET["Unmet Demand"] -->|"+ residual fill"| IM
        IM_REDUCE -->|"-"| IM
    end

    subgraph "Balance of Payments (BPM6)"
        EX_OIL -->|"+"| EX_TOTAL["Total Exports EX_(t)"]
        EX_NONOIL -->|"+"| EX_TOTAL
        EX_TOTAL -->|"+"| TRADE_BAL["Trade Balance"]
        IM -->|"-"| TRADE_BAL
        NFA_GOV_LAG["NFA_gov(t-1)"] -->|"+ r_reserve"| NFA_INC["NFA Income"]

        NFA_INC -->|"+ extraction rate e"| PRIM_INC["Primary Income Balance"]

        FDI_STOCK_LAG["FDI_net_stock(t-1)"] -->|"+ r_FDI"| FDI_INC["FDI Income Payments"]
        FDI_INC -->|"-  repatriation"| PRIM_INC
        BOND_EXT_INT["Ext Bond Interest"] -->|"-"| PRIM_INC
        REMIT["Remittances"] -->|"-"| SEC_INC["Secondary Income Balance"]
        TRADE_BAL -->|"+"| CURR_ACC["Current Account"]
        PRIM_INC -->|"+"| CURR_ACC
        SEC_INC -->|"+"| CURR_ACC
        CURR_ACC -->|"+"| DELTA_NFA["delta NFA_gov"]
        NFA_EXTRACT["NFA Income Reinvested (1-e)"] -->|"+"| DELTA_NFA
        GOV_PIF["Gov PIF Outflow"] -->|"-"| DELTA_NFA
        BOND_EXT_ISS["Ext Bond Issuance"] -->|"+"| DELTA_NFA
        FDI_NET["FDI Net Inflow"] -->|"+"| DELTA_NFA
        NFA_GOV_LAG -->|"+ carry"| NFA_GOV
        DELTA_NFA -->|"+"| NFA_GOV["NFA_gov(t)"]
    end
```

## 5. Supply-Demand Reconciliation & Leontief Production
model.py §6–10: expectations, aggregate demand, Leontief production function, unmet demand.

```mermaid
graph TB
    subgraph "Supply Decision  -  Section 6"
        gY_e["Expected Demand Growth gY_e"] -->|"+"| Qs["Supply Choice Q_s_(t)"]
        Y_lag["Y_(t-1) Realized GDP"] -->|"+"| Qs
        Y_TS["Y Time Series AR Model"] -->|"+ forecast"| gY_e
        GOV_BOOST["Gov Confidence Boost"] -->|"+"| Qs
        Qs -->|"+ nominal to real"| qs["Real Supply q_s"]
    end

    subgraph "Aggregate Demand  -  Section 6.2"
        C["Consumption C"] -->|"+"| Qd["Demand Q_d_(t)"]
        GY["Gov Spending GY"] -->|"+"| Qd
        I_DEM["Investment Demand I_demand"] -->|"+"| Qd
        EX_OIL["Oil Exports"] -->|"+"| Qd
        EX_NONOIL["Non-Oil Exports"] -->|"+"| Qd
        IM["Imports (subtract)"] -->|"-"| Qd
    end

    subgraph "Leontief Production  -  Section 7"
        BETA["beta * IntP(t-1)"] -->|"+ plus flex"| MIN1["Min: IntP capacity vs K capacity"]
        KAPPA["kappa * K(t-1)"] -->|"+ times flex_capital"| MIN1
        ALPHA["alpha * W(t)"] -->|"+"| MIN2["Min: Labor capacity vs Min1"]
        MIN1 -->|"+"| MIN2
        Qs -->|"+"| Ys["Realized Supply Y_s_(t)"]
        MIN2 -->|"-  binding min"| Ys
    end

    subgraph "Binding Constraints  -  Section 7.1"
        Qs -->|"-  gap"| GAP_INT["IntP Binding Gap"]
        BETA -->|"+ capacity"| GAP_INT
        Qs -->|"-  gap"| GAP_K["Capital Binding Gap"]
        KAPPA -->|"+ capacity"| GAP_K
        Qs -->|"-  gap"| GAP_W["Wage Binding Gap"]
        ALPHA -->|"+ capacity"| GAP_W
    end

    subgraph "Unmet Demand and Reconciliation  -  Section 8"
        Qd -->|"+"| UNMET["Unmet Demand = Q_d - Y_s"]
        Ys -->|"-"| UNMET
        UNMET -->|"+ option 1"| IM_ADJ["Increase Imports"]
        UNMET -->|"-  option 2"| REDUCE_EX["Reduce Non-Oil Exports"]
        UNMET -->|"-  option 2"| REDUCE_GY["Reduce Gov Consumption"]
        UNMET -->|"-  option 2"| REDUCE_C["Reduce HH Consumption"]
        UNMET -->|"-  option 2"| REDUCE_I["Reduce Investment"]
    end

    subgraph "IO Structure and Output  -  Section 9-10"
        Ys -->|"+"| Y_FINAL["Realized Y_(t)"]
        Y_FINAL -->|"+ Leontief"| X["Total Output X = inv(I-A) * Y"]
        A["A Matrix"] -->|"+"| X
        V2030["Vision 2030 Policy"] -->|"+ direct sector growth"| Y_FINAL
        V2030 -->|"+ cluster supply chain growth"| Y_FINAL
        V2030 -->|"-  import substitution"| IM
    end

    GAP_INT -->|"+ scarcity signal"| I_ENDOG["Endogenous Investment Growth"]
    GAP_K -->|"+ scarcity signal"| I_ENDOG
```

```

    subgraph "Leontief Production  -  Section 7"    GAP_K -->|scarcity signal| I_ENDOG

        BETA["beta * IntP(t-1)"] -->|plus flex| MIN1["Min: IntP capacity vs K capacity"]    GAP_INT -->|scarcity signal| I_ENDOG["Endogenous Investment Growth"]

        KAPPA["kappa * K(t-1)"] -->|times flex_capital| MIN1

        ALPHA["alpha * W(t)"] --> MIN2["Min: Labor capacity vs Min1"]    end

        MIN1 --> MIN2        V2030 -->|import substitution| IM

        Qs --> Ys["Realized Supply Y_s_(t)"]        V2030 -->|cluster supply chain growth| Y_FINAL

        MIN2 --> Ys        V2030["Vision 2030 Policy"] -->|direct sector growth| Y_FINAL

    end        A["A Matrix"] --> X

        Y_FINAL --> X["Total Output X = inv(I-A) * Y"]

    subgraph "Binding Constraints  -  Section 7.1"        Ys --> Y_FINAL["Realized Y_(t)"]

        Qs -->|gap| GAP_INT["IntP Binding Gap"]    subgraph "IO Structure and Output  -  Section 9-10"

        BETA --> GAP_INT

        Qs -->|gap| GAP_K["Capital Binding Gap"]    end

        KAPPA --> GAP_K        UNMET -->|option 2| REDUCE_I["Reduce Investment"]

        Qs -->|gap| GAP_W["Wage Binding Gap"]        UNMET -->|option 2| REDUCE_C["Reduce HH Consumption"]

        ALPHA --> GAP_W        UNMET -->|option 2| REDUCE_GY["Reduce Gov Consumption"]

    end        UNMET -->|option 2| REDUCE_EX["Reduce Non-Oil Exports"]

        UNMET -->|option 1| IM_ADJ["Increase Imports"]

    subgraph "Unmet Demand and Reconciliation  -  Section 8"        Ys --> UNMET
        Qd --> UNMET["Unmet Demand = Q_d - Y_s"]

## 6. Energy Module
energy_module.py: energy demand, generation mix, emissions, net zero, carbon market.

```mermaid
graph TB
    subgraph "Energy Demand  -  Section 1-2"
        X["Total Output X_(t)"] -->|"+ gas intensity"| GAS["Gas Use_(t)"]
        X -->|"+ oil intensity"| OIL["Oil Use_(t)"]
        X -->|"+ elec intensity"| ELEC["Electricity Use_(t)"]
        EFF["Energy Reduction Factor"] -->|"-  decreases demand"| GAS
        EFF -->|"-  decreases demand"| OIL
        EFF -->|"-  decreases demand"| ELEC
        OIL_RED["Oil-to-Gas Switching V2030"] -->|"-  reduces oil"| OIL
        OIL_RED -->|"+ adds to gas"| GAS
    end

    subgraph "Electrification Strategy"
        ELECTR_RATE["Annual Electrification Rate"] -->|"+ cumulative shift"| FOSSIL_SHIFT["Fossil to Electricity Shift"]
        FOSSIL_SHIFT -->|"-  replaces"| OIL
        FOSSIL_SHIFT -->|"-  replaces"| GAS
        FOSSIL_SHIFT -->|"+ adds load"| ELEC
    end

    subgraph "Hydrogen Module"
        K_H2["H2 Physical Capital"] -->|"+ capacity"| H2_PROD["H2 Production"]
        H2_PROD -->|"+ efficiency factor"| RE_USED_H2["RE Used for H2"]
        H2_PROD -->|"+ export share"| H2_EXPORT["H2 Export"]
        H2_PROD -->|"+ domestic share"| H2_DOM["H2 Domestic Use"]
        H2_DOM -->|"-  substitutes"| GAS
    end

    subgraph "Electricity Generation Mix  -  Section 3"
        ELEC -->|"+ total demand"| TOTAL_ELEC["Total Elec Generation"]
        RE_GEN["RE Generation Capacity"] -->|"+ net of H2 use"| RE_ELEC["RE for Electricity"]
        RE_ELEC -->|"-  displaces fossil"| FF_GAP["Fossil Fuel Gap"]
        FF_GAP -->|"+ oil share declining"| OIL_ELEC["Oil for Electricity"]
        FF_GAP -->|"+ gas share rising"| GAS_ELEC["Gas for Electricity"]
    end

    subgraph "Fossil Substitution by Surplus RE  -  Section 4"
        RE_SURPLUS["Surplus RE after Elec"] -->|"-  substitutes"| OIL_NONEL["Non-Elec Oil Use"]
        RE_SURPLUS -->|"-  substitutes"| GAS_NONEL["Non-Elec Gas Use"]
    end

    subgraph "Emissions  -  Section 5"
        OIL -->|"+ emission factor"| OIL_EMIS["Oil Emissions"]
        GAS -->|"+ emission factor"| GAS_EMIS["Gas Emissions"]
        ELEC -->|"+ dynamic factor"| ELEC_EMIS["Electricity Emissions"]
        OIL_EMIS -->|"+"| GROSS_EMIS["Gross Emissions"]
        GAS_EMIS -->|"+"| GROSS_EMIS
        ELEC_EMIS -->|"+"| GROSS_EMIS
    end

    subgraph "Net Zero Offsets"
        CCU["CCU Capacity"] -->|"+"| REUSE["Flow: Reuse/Recycle"]
        CCS["CCS Ramp 2035 target"] -->|"+"| REMOVE_CCS["Flow: Remove CCS"]
        SGI["Saudi Green Initiative"] -->|"+"| REMOVE_NAT["Flow: Remove Nature"]
        REUSE -->|"+"| PHYS_REMOVAL["Physical Removal"]
        REMOVE_CCS -->|"+"| PHYS_REMOVAL
        REMOVE_NAT -->|"+"| PHYS_REMOVAL
        GROSS_EMIS -->|"+"| NET_EMIS["Net Emissions"]
        PHYS_REMOVAL -->|"-  subtracts"| NET_EMIS
    end

    subgraph "Carbon Market"
        NET_EMIS -->|"+"| DEMAND_CC["Carbon Credit Demand"]
        REMOVE_CCS -->|"+"| SUPPLY_CC["Carbon Credit Supply"]
        REMOVE_NAT -->|"+"| SUPPLY_CC
        DEMAND_CC -->|"+"| TRADED["Traded Volume"]
        SUPPLY_CC -->|"+"| TRADED
        TRADED -->|"-  cost to firms"| CC_COST["Carbon Credit Cost to P_(t)"]
        TRADED -->|"+ revenue to producers"| CC_REV["Carbon Credit Revenue to P_(t)"]
    end
```

## 7. Stock-Flow Accounting
model.py §12: GDP, capital dynamics, profits, government, banks, households, sovereign wealth.

```mermaid
graph TB
    subgraph "12.1 GDP and Growth"
        Y["Y_(t) = C + GY + I_demand + EX - IM"]
        Y -->|"+ deflate"| y["Real GDP y_(t)"]
        Y -->|"+ oil sectors mask"| Y_OIL["Oil GDP"]
        Y -->|"+ non-oil mask"| Y_NONOIL["Non-Oil GDP"]
        Y -->|"+ pct change"| gY["GDP Growth Rate gY"]
        X["Total Output X"] -->|"+"| dX["Output Shares dX_(t)"]
    end

    subgraph "12.2 Capital Dynamics"
        K_lag["K_(t-1)"] -->|"-  depreciate delta"| K["K_(t) = K(t-1)(1-delta) + I(t-1)"]
        I_lag["I_(t-1)"] -->|"+"| K
        K -->|"+ utilization"| u["Capacity Utilization u"]
        x["Real output x"] -->|"+"| u
    end

    subgraph "12.3 Profits"
        X2["X_(t)"] -->|"+"| P["Profits P_(t)"]
        IntP["Intermediate Purchases"] -->|"-"| P
        W["Wages W_(t)"] -->|"-"| P
        TAX_PROD["Production Taxes"] -->|"-"| P
        TAX_PROD_NET["Product Taxes net"] -->|"-"| P
        SUB["Production Subsidies"] -->|"+"| P
        TAX_INV["Investment Taxes"] -->|"-"| P
        CC_COST["Carbon Credit Costs"] -->|"-"| P
        CC_REV["Carbon Credit Revenue"] -->|"+"| P
    end

    subgraph "12.4 Government"
        VAT["VAT = C * tau_vat"] -->|"+"| GOV_REV["Gov Revenue"]
        TH["HH Taxes on profits"] -->|"+"| GOV_REV
        GP["Public Profits"] -->|"+"| GOV_REV
        TAX_w["Wage Tax"] -->|"+"| GOV_REV
        TAX_p["Profit Tax"] -->|"+"| GOV_REV
        TAX_PROD2["Production/Product Taxes"] -->|"+"| GOV_REV
        NFA_EXTRACT["NFA Income (extracted share e)"] -->|"+"| GOV_REV
        GY["Gov Consumption"] -->|"+"| GOV_EXP["Gov Expenditure"]
        SUB2["Subsidies"] -->|"+"| GOV_EXP
        GI["Gov Investment"] -->|"+"| GOV_EXP
        BOND_INT["Dom Bond Interest+Repayment"] -->|"+"| GOV_EXP
        EXT_BOND_SRV["Ext Bond Service"] -->|"+"| GOV_EXP
        GOV_EXP -->|"+ deficit"| BOND["Dom Bonds B(t)"]
        GOV_REV -->|"-  surplus"| BOND
        GOV_REV -->|"+"| NET_W["Gov Net Wealth"]
        GOV_EXP -->|"-"| NET_W
    end

    subgraph "12.5 Firms and Banks (Endogenous Money)"
        I_PRIV["Private Investment"] -->|"+ loan share"| L["Loans L_(t)"]
        L_lag["L_(t-1)"] -->|"-  repayment rho"| L
        L -->|"= endogenous money"| D_HH["HH Deposits D_HH = L"]
        L -->|"+ interest rl"| PB["Bank Profits PB"]
        BOND3["Gov Dom Bonds"] -->|"+ interest rm"| PB
        V2["HH Wealth V"] -->|"+"| PB
    end

    subgraph "12.6 Household Accounting"
        W2["Wages"] -->|"+"| YD_W["Disposable Wage Income YD_wage"]
        TAX_W2["Wage Tax"] -->|"-"| YD_W
        REMIT["Remittances"] -->|"-  outflow"| YD_W
        P2["Private Profits distributed"] -->|"+"| YD_P["Disposable Profit Income YD_profit"]
        PB2["Bank Profits to HH"] -->|"+"| YD_P
        TH2["HH Taxes TH"] -->|"-"| YD_P
        LOAN_INT["Loan Interest"] -->|"-"| YD_P
        FDI_INC["FDI Income Payments"] -->|"-  repatriation"| YD_P
        YD_W -->|"+"| V["HH Wealth V(t)"]

        YD_P -->|"+"| V
        C2["Consumption"] -->|"-"| V
        VAT2["VAT"] -->|"-"| V
        I_PROF["Investment from profits"] -->|"-"| V
        REMIT2["Remittances"] -->|"-"| V
        D_HH2["HH Deposits D_HH"] -->|"+ asset"| HH_NW["HH Total Net Worth"]
        V -->|"+ portfolio claims"| HH_NW
        E_FIRM_HH["Firm Equity HH share E_firm_HH"] -->|"+"| HH_NW
        E_BANK_HH["Bank Equity HH share E_bank_HH"] -->|"+"| HH_NW
    end

    subgraph "12.7 Sovereign Wealth & External Sector"
        NFA_GOV_LAG["NFA_gov(t-1)"] -->|"+ r_reserve"| NFA_INC["NFA Income"]
        NFA_INC -->|"+ extraction e"| NFA_EXTRACT
        NFA_INC -->|"- reinvested (1-e)"| NFA_REINV["NFA Reinvested"]
        NFA_REINV -->|"+"| NFA_GOV["NFA_gov(t)"]
        NFA_GOV_LAG -->|"+ carry"| NFA_GOV
        PIF_OUT -->|"-"| NFA_GOV
        GOV_SURPLUS["Gov Fiscal Surplus"] -->|"+ pif_rate"| PIF["PIF Asset Stock"]
        GOV_REV2["Gov Revenue"] -->|"+"| GOV_SURPLUS
        GOV_EXP2["Gov Expenditure"] -->|"-"| GOV_SURPLUS
        PIF -->|"+ foreign invest"| PIF_OUT["PIF Outflow"]
        NFA_GOV -->|"-  below floor"| BOND_EXT_TRIG["Ext Bond Trigger"]
        BOND_EXT_TRIG -->|"+"| BOND_EXT["Ext Bond Stock B_ext"]
        BOND_EXT -->|"+ debt service"| EXT_BOND_SRV
        OFA_HH["OFA_HH"] -->|"+ TFM residual"| OFA_SUM["Sum OFA = 0 (closure)"]
        OFA_GOV["OFA_Gov"] -->|"+"| OFA_SUM
        OFA_FIRMS["OFA_Firms"] -->|"+"| OFA_SUM
        OFA_BANKS["OFA_Banks"] -->|"+"| OFA_SUM
        OFA_ROW["OFA_ROW"] -->|"+"| OFA_SUM
    end
```

## 8. IO Table & Endogenous A Matrix
calc_io_table (model.py) + endogenize_input_output_matrix_A.py

```mermaid
graph TB
    subgraph "IO Table Dynamics  -  calc_io_table"
        K_DESAL["Desal Capital K_desal(t-1)"] -->|"+"| DESAL_OUT["Desal Output = K * u * eK"]
        X_WATER["Water Sector Output x_water(t-1)"] -->|"-  normalise"| DESAL_SHARE["Desal Share = Desal_Out / x_water"]
        DESAL_OUT -->|"+"| DESAL_SHARE
        DESAL_SHARE -->|"-  reduces other inputs"| A_WATER["A col water * (1 - desal_share)"]
        DESAL_SHARE -->|"+ adds desal row"| A_DESAL["A desal water = desal_share"]
        A_WATER -->|"+"| A_NEW["Updated A Matrix"]
        A_DESAL -->|"+"| A_NEW
    end

    subgraph "Endogenize IO Matrix A  -  Vision 2030"
        EXCESS_IMP["Excess Import Measure"] -->|"+"| RANK["Sector Import Dependency Rank"]
        SUPPLY_SCARCITY["Supply Constraint Metrics"] -->|"+"| RANK

        subgraph "Step 1: Direct V2030 Sector Growth"
            V2030_RATES["V2030 Sectoral Growth Rates"] -->|"+"| Y_INCREASE["Increase Y_(t) for V2030 sectors"]
            Y_INCREASE -->|"-  substitution"| IM_DECREASE["Decrease Imports of V2030 sectors"]
            Y_INCREASE -->|"+"| gI_INCREASE["Increase Endogenous Investment Growth"]
        end

        subgraph "Step 2: Cluster Supply Chain Growth"
            V2030_RATES2["V2030 Growth Rates"] -->|"+"| INTP_PROXY["Increase IntP for V2030 columns"]
            INTP_PROXY -->|"+"| SUPPLIER_INCREASE["Supplier Sector Y_(t) Increase"]
            SUPPLIER_INCREASE -->|"-  substitution"| IM_SUPPLIER["Decrease Supplier Imports"]
            SUPPLIER_INCREASE -->|"+"| gI_SUPPLIER["Increase Supplier Investment Growth"]
        end
    end

    subgraph "Output Calculation"
        A_NEW -->|"+ Leontief inverse"| X_CALC["X = inv(I - A) * Y  Leontief inverse"]
        Y_FINAL["Realized Y_(t)"] -->|"+"| X_CALC
        X_CALC -->|"+"| Z["Z Matrix = A * diag(X)"]
        X_CALC -->|"+"| INTP["IntP = sum(A,0) * X"]
        X_CALC -->|"+"| INTS["IntS = A * X"]
    end
```

## 9. Water Module
water_module.py: crop demand, irrigation, water balance, cost accounting.

```mermaid
graph TB
    subgraph "Water Module"
        subgraph "Crop Demand Model"
            POP["Population SSP growth"] -->|"+"| FOOD_DEM["Per Capita Food Demand"]
            SSR["Self-Sufficiency Ratios scenario"] -->|"+"| DOM_PROD["Domestic Production Required"]
            FOOD_DEM -->|"+"| DOM_PROD
            YIELD["Crop Yield ton per m2"] -->|"-  fewer area needed"| AREA["Cultivated Area"]
            DOM_PROD -->|"+"| AREA
        end

        subgraph "Irrigation Technology"
            AREA -->|"+"| IRR["Irrigation Water Demand"]
            BWF["Blue Water Footprint coeff."] -->|"+"| IRR
            TECH_MIX["Technology Mix Surface/Sprinkler/Drip"] -->|"+"| EFF["Irrigation Efficiency"]
            EFF -->|"-  reduces demand"| IRR
            GH_PCT["Greenhouse Production pct"] -->|"-  reduces field area"| IRR
        end

        subgraph "Water Supply"
            K_DESAL["Desal Capital Stock"] -->|"+"| POT_DESAL["Desalination Potential"]
            K_WWATER["Wastewater Capital"] -->|"+"| POT_WWATER["Wastewater Potential"]
            POT_DESAL -->|"+"| TOTAL_SUPPLY["Total Water Supply"]
            POT_WWATER -->|"+"| TOTAL_SUPPLY
            GROUND["Groundwater residual"] -->|"+"| TOTAL_SUPPLY
        end

        subgraph "Water Balance"
            IRR -->|"+"| WATER_USE["Total Water Use National"]
            TOTAL_SUPPLY -->|"-  closes gap"| GAP["Water Supply-Demand Gap"]
            WATER_USE -->|"+"| GAP
            GAP -->|"+ if gap positive"| I_DESAL_SIGNAL["Signal: Increase Desal Investment"]
            GAP -->|"+ if gap positive"| I_WWATER_SIGNAL["Signal: Increase Wwater Investment"]
        end

        subgraph "Cost Accounting"
            AREA2["Irrigated Area"] -->|"+"| INVEST_COST["Irrigation Investment Cost"]
            TECH_MIX2["Technology Mix"] -->|"+"| INVEST_COST
            AREA2 -->|"+"| OM_COST["O and M Cost"]
            SUBSIDY_RATE["Subsidy Rate scenario"] -->|"-  reduces"| NET_COST["Net Cost after Subsidy"]
            INVEST_COST -->|"+"| NET_COST
            OM_COST -->|"+"| NET_COST
        end

        subgraph "Nature-Energy-Water Nexus"
            SGI["Saudi Green Initiative trees"] -->|"+ 0.315 Gt water per Mt CO2"| SGI_WATER["SGI Water Demand"]
            SGI_WATER -->|"+"| WATER_USE
        end
    end
```

## 10. Government Sector
model.py §5.2 & §12.4: revenue, expenditure, bonds, countercyclical policy.

```mermaid
graph TB
    subgraph "Government Sector"
        subgraph "Government Revenue"
            C["Consumption C"] -->|tau_vat| VAT["VAT"]
            P_PRIV["Private Profits"] -->|tau_HH| TH["HH Taxes"]
            P_PUB["Public Profits oil rents"] --> GP["Gov Oil Profits"]
            W["Wages"] -->|wage tax rate| TAX_W["Wage Tax"]
            P_PRIV2["Private Profits"] -->|profit tax rate| TAX_P["Profit Tax"]
            X["Output X"] -->|tau_production| TAX_PROD["Production Tax"]
            X -->|tau_products| TAX_PROD_NET["Product Tax net"]
            VAT --> GOV_REV["Total Gov Revenue"]
            TH --> GOV_REV
            GP --> GOV_REV
            TAX_W --> GOV_REV
            TAX_P --> GOV_REV
            TAX_PROD --> GOV_REV
            TAX_PROD_NET --> GOV_REV
        end

        subgraph "Government Expenditure"
            GY_TS["GY Time Series AR Model"] -->|or avg growth| GY["Gov Consumption GY"]
            OIL_SHOCK["Oil Export Shock lag 1"] -->|cyclicality factor| GY
            GOV_BOOST["Confidence Boost"] --> GY
            GY -->|sectoral dG| GY_SECT["Sectoral GY_(t)"]
            X2["Output X"] -->|subsidy rate| SUB["Production Subsidies"]
            GI["Public Investment"] --> GOV_EXP["Total Gov Expenditure"]
            GY --> GOV_EXP
            SUB --> GOV_EXP
            BOND_INT["Bond Interest plus Repayment"] --> GOV_EXP
            RESID["Gov Expenditure Residual"] --> GOV_EXP
        end

        subgraph "Government Balance Sheet"
            GOV_EXP --> BOND["Gov Bonds B(t)"]
            GOV_REV --> BOND
            B_lag["Bonds B(t-1)"] -->|repayment rho| BOND
            GOV_REV --> NET_W["Gov Net Wealth"]
            GOV_EXP --> NET_W
            ARAMCO["Aramco Market Cap"] --> NET_W_PLUS["Gov Net Wealth plus Aramco"]
            NET_W --> NET_W_PLUS
        end

        subgraph "Countercyclical Policy"
            gY_neg["Negative GDP growth"] --> BOOST["Gov Confidence Boost"]
            INF_neg["Negative Inflation"] --> BOOST
            BOOST -->|factor| I_PUB_BOOST["Boost Public Investment"]
            BOOST --> GY
            BOOST --> Qs["Supply Choice Q_s"]
        end
    end
```

## 11. FDI Module
FDI_module.py: foreign direct investment decomposition and balance of payments.

```mermaid
graph TB
    subgraph "FDI Module"
        subgraph "FDI Growth Modes"
            MODE_A["Mode A: Follow Domestic Growth"] --> gI_REAL["Sectoral I growth = I(t)/I(t-1) - 1"]
            gI_REAL --> FDI_SECT["Sectoral FDI Growth"]
            FDI_TARGET["FDI Target V2030 window"] -->|premium| FDI_SECT
            FDI_SECT -->|cap 50 pct per sector| FDI_NET["FDI Net Flow_(t)"]

            MODE_B["Mode B: Fixed Growth Rate"] --> FDI_FIXED["FDI = Base * (1+target)^t"]
            FDI_FIXED -->|V2030 timing cap| FDI_NET
        end

        subgraph "FDI Accounting"
            FDI_NET -->|distributed by I shares| FDI_FLOW["FDI_net_flow_(t)"]
            FDI_FLOW --> FDI_GROSS["FDI Gross Flow"]
            FDI_FLOW --> FDI_STOCK["FDI Stock cumulative"]
            FDI_FLOW --> SHARE["FDI Share of Investment"]
        end

        subgraph "Impact on Domestic Financing"
            I_TOTAL["Total Investment I_(t)"] --> I_DOM["I_domestic = I - FDI_flow"]
            FDI_FLOW --> I_DOM
            I_DOM -->|private share| I_PRIV_DOM["I_private_dom needs loans"]
            I_DOM -->|public share| I_PUB_DOM["I_public_dom gov budget"]
        end

        subgraph "Balance of Payments"
            FDI_NET --> CAP_ACC["Capital Account"]
            EX["Exports"] --> CURR_ACC["Current Account = EX - IM"]
            IM["Imports"] --> CURR_ACC
        end
    end
```

## 12. Key Feedback Loops (Reinforcing & Balancing)
Cross-cutting loops across all modules. **R** = reinforcing (positive), **B** = balancing (negative).

```mermaid
graph TB
    subgraph "R1: Multiplier-Accelerator Loop (positive)"
        Y1["GDP Y UP"] --> C1["Consumption C UP"]
        C1 --> Qd1["Demand Q_d UP"]
        Qd1 --> Y1
        Y1 --> I1["Investment I UP"]
        I1 --> Qd1
    end
```

```mermaid
graph TB
    subgraph "R2: Profit-Investment Loop (positive)"
        P1["Profits P UP"] --> YD1["HH Profit Income UP"]
        YD1 --> C2["Consumption UP"]
        C2 --> X1["Output X UP"]
        X1 --> P1
        P1 --> I2["Investment from profits UP"]
        I2 --> K1["Capital K UP"]
        K1 --> Y_S["Supply Capacity UP"]
    end
```

```mermaid
graph TB
    subgraph "R3: Wage-Price Spiral (positive)"
        INF1["Inflation UP"] --> W1["Wages UP"]
        W1 --> COST["Unit Costs UP"]
        COST --> P_PRICE["Prices UP"]
        P_PRICE --> INF1
    end
```

```mermaid
graph TB
    subgraph "B1: Supply Constraint Balancing (negative)"
        Y_GROW["GDP Growth UP"] --> Q_S["Supply Choice UP"]
        Q_S --> LEONT["Production Leontief min"]
        K_LAG["Capital K(t-1)"] --> LEONT
        LEONT -->|binding constraint| UNMET["Unmet Demand UP"]
        UNMET -->|import increase| IM1["Imports UP"]
        IM1 -->|reduces GDP| Y_DOWN["GDP DOWN"]
    end
```

```mermaid
graph TB
    subgraph "B2: Government Stabilizer (negative)"
        gY_NEG["GDP Growth below 0"] --> GOV_BOOST2["Countercyclical Boost"]
        GOV_BOOST2 --> GY2["Gov Spending UP"]
        GOV_BOOST2 --> I_PUB["Public Investment UP"]
        GY2 --> DEM["Demand UP"]
        I_PUB --> DEM
        DEM --> Y_RECOV["GDP Recovery UP"]
    end
```

```mermaid
graph TB
    subgraph "R4: Energy Transition Loop (positive)"
        I_RE["RE Investment UP"] --> RE_CAP["RE Capacity UP"]
        RE_CAP --> RE_SHARE["RE Share UP"]
        RE_SHARE --> FOSSIL_DOWN["Fossil Use DOWN"]
        FOSSIL_DOWN --> EMIS_DOWN["Emissions DOWN"]
        EMIS_DOWN --> CC_COST_DOWN["Carbon Credit Cost DOWN"]
        CC_COST_DOWN --> PROFITS_UP["Profits UP"]
        PROFITS_UP --> I_RE
    end
```

```mermaid
graph TB
    subgraph "B3: Water-Energy-Food Nexus (negative)"
        POP_GROW["Population Growth"] --> WATER_DEM["Water Demand UP"]
        SGI_TREES["SGI Tree Planting"] --> WATER_DEM
        WATER_DEM --> GAP["Water Gap UP"]
        GAP --> I_DESAL2["Desal Investment UP"]
        I_DESAL2 --> ENERGY_DEM["Energy Demand UP"]
        I_DESAL2 --> WATER_SUP["Water Supply UP"]
        WATER_SUP --> GAP
    end
```

## 13. Balance of Payments & Sovereign Wealth
Full BPM6 current account structure, NFA_gov dynamics, PIF, and two-phase sovereign financing (FDI_module.py + model.py §12.7).

```mermaid
graph TB
    subgraph "Current Account BPM6"
        EX_CA["Total Exports EX_(t)"] -->|"+"| TRADE_BAL_CA["Trade Balance"]
        IM_CA["Total Imports IM_(t)"] -->|"-"| TRADE_BAL_CA
        NFA_GOV_CA["NFA_gov(t-1)"] -->|"+  r_reserve"| NFA_INC_CA["NFA Income"]
        NFA_INC_CA -->|"+  extraction e"| PRIM_IN["Primary Income (+)"]
        FDI_STCK_CA["FDI Net Stock(t-1)"] -->|"+  r_FDI"| FDI_INC_CA["FDI Income Payments"]
        FDI_INC_CA -->|"-"| PRIM_IN
        BOND_EXT_CA["Ext Bond Stock"] -->|"+  interest"| EXT_INT_CA["Ext Bond Interest"]
        EXT_INT_CA -->|"-"| PRIM_IN
        REMIT_CA["Remittances"] -->|"-"| SEC_IN["Secondary Income"]
        TRADE_BAL_CA -->|"+"| CA["Current Account"]
        PRIM_IN -->|"+"| CA
        SEC_IN -->|"+"| CA
    end

    subgraph "Financial Account"
        FDI_FA["FDI Net Inflow"] -->|"+"| FA["Financial Account"]
        BOND_ISS_FA["Routine Ext Bond Issuance"] -->|"+"| FA
        EMRG_BONDS["Emergency Ext Bonds Phase 2"] -->|"+"| FA
        DELTA_RES["delta Reserves = -delta NFA_gov"] -->|"-"| FA
        CA -->|"+  closure"| BOP_CHECK["BoP Check = 0"]
        FA -->|"-  closure"| BOP_CHECK
    end

    subgraph "NFA_gov Dynamics"
        CA -->|"+"| DNFA["delta NFA_gov"]
        FDI_FA -->|"+"| DNFA
        BOND_ISS_FA -->|"+"| DNFA
        NFA_INC_REINV["NFA Income Reinvested (1-e)"] -->|"+"| DNFA
        PIF_OUT_NFA["PIF Outflow foreign invest"] -->|"-"| DNFA
        DNFA -->|"+"| NFA_GOV_T["NFA_gov(t)"]
        NFA_GOV_CA -->|"+  carry"| NFA_GOV_T
    end

    subgraph "PIF and Sovereign Wealth Accumulation"
        GOV_SURP_SW["Gov Fiscal Surplus"] -->|"+  gov_surplus_pif_rate"| PIF_SW["PIF Asset Stock"]
        PIF_SW -->|"+  foreign invest"| PIF_OUT_NFA
        NFA_GOV_T -->|"+  returns r_reserve"| NFA_INC_CA
        NFA_INC_CA -->|"+  extracted e"| GOV_REV_SW["Gov Revenue"]
        NFA_INC_CA -->|"+  reinvested 1-e"| NFA_INC_REINV
    end

    subgraph "Two-Phase Sovereign Financing"
        NFA_GOV_T -->|"-  floor 0.5*NFA_init"| PHASE_CHECK["NFA Floor Check"]
        PHASE_CHECK -->|"+  Phase 1: NFA above floor"| RESERVE_DRAW["Use Reserves drawdown"]
        RESERVE_DRAW -->|"-"| NFA_GOV_T
        PHASE_CHECK -->|"+  Phase 2: NFA below floor"| EMRG_ISSUE["Emergency Bond Issuance"]
        EMRG_ISSUE -->|"+"| EMRG_BONDS
        EMRG_BONDS -->|"+"| EXT_BOND_STK["Ext Bond Stock B_ext"]
        EXT_BOND_STK -->|"+  interest+repay"| EXT_DEBT_SRV2["Ext Debt Service"]
        EXT_DEBT_SRV2 -->|"+"| GOV_EXP_SW["Gov Expenditure"]
        GOV_EXP_SW -->|"-"| GOV_SURP_SW
    end
```

## Export Options

**Option A — HTML:** Run the cell below to export all diagrams into one interactive HTML file.

**Option B — PNG/SVG images for LaTeX:** Run the second cell below to save each diagram as a separate image file.

**Option C — CSV/JSON for kumu.io:** Run the third cell below to export nodes and edges for import into [kumu.io](https://kumu.io).

| Format | LaTeX command | Notes |
|--------|--------------|-------|
| PNG | `\includegraphics[width=\textwidth]{Figures/01_1_overall_model_architecture.png}` | Universally supported |
| SVG | `\includesvg[width=\textwidth]{Figures/01_1_overall_model_architecture.svg}` | Vector (sharper), requires `\usepackage{svg}` + Inkscape |

| kumu.io file | Contents | Import step |
|--------------|----------|-------------|
| `kumu_elements.csv` | Label, Type (module grouping) | Import → Spreadsheet → Elements |
| `kumu_connections.csv` | From, To, Type (edge label) | Import → Spreadsheet → Connections |
| `kumu_data.json` | Combined elements + connections | Import → JSON |

In [6]:
# Option A: Export all diagrams to a single HTML file
export_all_to_html()

Exported 38 diagrams to Figures/causal_loop_diagrams.html


In [7]:
# Export each diagram as a separate PNG (for LaTeX \includegraphics)
# Change fmt="svg" for vector graphics, or theme="neutral" for a cleaner look.
# THIS takes a LONG time, set this to False if you just want a quick export
long_export = False
if long_export:
    export_diagrams_to_images(fmt="png", theme="default")

  [ 1/38] ERROR: 1. Overall Model Architecture — ReadTimeout: HTTPSConnectionPool(host='kroki.io', port=443): Read timed out. (read timeout=60)
  [ 2/38] ERROR: 2. Prices, Wages & Consumption — ReadTimeout: HTTPSConnectionPool(host='kroki.io', port=443): Read timed out. (read timeout=60)
  [ 3/38] ERROR: 3. Investment Module — ReadTimeout: HTTPSConnectionPool(host='kroki.io', port=443): Read timed out. (read timeout=60)
  [ 4/38] ERROR: 4. Exports, Imports, Green Exports & Balance of Payments — ReadTimeout: HTTPSConnectionPool(host='kroki.io', port=443): Read timed out. (read timeout=60)
  [ 5/38] ERROR: 5. Supply-Demand Reconciliation & Leontief Production — ReadTimeout: HTTPSConnectionPool(host='kroki.io', port=443): Read timed out. (read timeout=60)
  [ 6/38] ERROR: 6. Energy Module — ReadTimeout: HTTPSConnectionPool(host='kroki.io', port=443): Read timed out. (read timeout=60)
  [ 7/38] ERROR: 7. Stock-Flow Accounting — ReadTimeout: HTTPSConnectionPool(host='kroki.io', port=443): R

In [8]:
# Option C: Export nodes & edges for kumu.io (CSV + JSON)
export_for_kumu()

Removed 22 stale kumu_* file(s) from Figures/
Writing per-diagram JSON files:
  CLD 1  (39 nodes, 50 edges)  ->  kumu_01_overall_model_architecture.json
  CLD 2  (22 nodes, 21 edges)  ->  kumu_02_prices_wages_consumption.json
  CLD 3  (44 nodes, 45 edges)  ->  kumu_03_investment_module.json
  CLD 4  (45 nodes, 47 edges)  ->  kumu_04_exports_imports_green_exports_balance_of_payments.json
  CLD 5  (33 nodes, 38 edges)  ->  kumu_05_supply_demand_reconciliation_leontief_production.json
  CLD 6  (39 nodes, 45 edges)  ->  kumu_06_energy_module.json
  CLD 7  (77 nodes, 76 edges)  ->  kumu_07_stock_flow_accounting.json
  CLD 8  (24 nodes, 21 edges)  ->  kumu_08_io_table_endogenous_a_matrix.json
  CLD 9  (29 nodes, 28 edges)  ->  kumu_09_water_module.json
  CLD 10  (33 nodes, 36 edges)  ->  kumu_10_government_sector.json
  CLD 11  (19 nodes, 17 edges)  ->  kumu_11_fdi_module.json
  CLD 12a  (4 nodes, 5 edges)  ->  kumu_12a_r1_multiplier_accelerator_loop_positive.json
  CLD 12b  (7 nodes, 7 edge

['Figures\\kumu_01_overall_model_architecture.json',
 'Figures\\kumu_02_prices_wages_consumption.json',
 'Figures\\kumu_03_investment_module.json',
 'Figures\\kumu_04_exports_imports_green_exports_balance_of_payments.json',
 'Figures\\kumu_05_supply_demand_reconciliation_leontief_production.json',
 'Figures\\kumu_06_energy_module.json',
 'Figures\\kumu_07_stock_flow_accounting.json',
 'Figures\\kumu_08_io_table_endogenous_a_matrix.json',
 'Figures\\kumu_09_water_module.json',
 'Figures\\kumu_10_government_sector.json',
 'Figures\\kumu_11_fdi_module.json',
 'Figures\\kumu_12a_r1_multiplier_accelerator_loop_positive.json',
 'Figures\\kumu_12b_r2_profit_investment_loop_positive.json',
 'Figures\\kumu_12c_r3_wage_price_spiral_positive.json',
 'Figures\\kumu_12d_b1_supply_constraint_balancing_negative.json',
 'Figures\\kumu_12e_b2_government_stabilizer_negative.json',
 'Figures\\kumu_12f_r4_energy_transition_loop_positive.json',
 'Figures\\kumu_12g_b3_water_energy_food_nexus_negative.json',

# KEN Model — Causal Loop Diagram Gallery

> **Mermaid Preview** — All diagrams below are rendered natively by VS Code's built-in Mermaid engine.  
> No code execution needed. Double-click any cell to edit the Mermaid source; click outside to re-render.

| # | Section |
|---|---------|
| 1 | Overall Model Architecture |
| 2 | Prices, Wages & Consumption |
| 3 | Investment Module |
| 4 | Exports, Imports, Green Exports & Balance of Payments |
| 5 | Supply-Demand Reconciliation & Leontief Production |
| 6 | Energy Module |
| 7 | Stock-Flow Accounting |
| 8 | IO Table & Endogenous A Matrix |
| 9 | Water Module |
| 10 | Government Sector |
| 11 | FDI Module |
| 12 | Key Feedback Loops (Reinforcing & Balancing) |
| 13 | Balance of Payments & Sovereign Wealth |

## 1. Overall Model Architecture
Top-level view of the entire KEN simulation loop (model.py).

```mermaid
graph TB
    subgraph "Demand Side (Section 3-5)"
        P_prices["Prices p_(t)"] -->|"+ cost push"| INF["Inflation"]
        INF -->|"+ indexation"| W["Wages W_(t)"]
        W -->|"+ MPC wage"| C["Consumption C(t)"]
        PROFITS_lag["Profits P(t-1)"] -->|"+ MPC profit"| C
        WEALTH_lag["HH Wealth V(t-1)"] -->|"+ MPC wealth"| C
        INF -->|"+ nominal adj"| C
        REMIT["Remittances"] -->|"-  income drain"| C
    end

    subgraph "Investment Module (Section 4)"
        I["Investment I_(t)"]
        I_energy["Energy Investment"]
        I_desal["Desal/Wwater Investment"]
        I_FDI["FDI Net Flows"]
        I_energy -->|"+"| I
        I_desal -->|"+"| I
        I -->|"+ base for FDI"| I_FDI
        I_FDI -->|"-  domestic share"| I_DOM["Domestic Investment"]
    end

    subgraph "Exports Module (Section 5)"
        EX_oil["Oil Exports"]
        EX_nonoil["Non-Oil Exports"]
        EX_green["Green Exports"]
        EX_oil -->|"+"| EX["Total Exports"]
        EX_nonoil -->|"+"| EX
        EX_green -->|"+ adds"| EX_nonoil
    end

    subgraph "Government (Section 5.2)"
        GY["Gov Consumption GY(t)"]
        GOV_REV["Gov Revenue"]
        GOV_EXP["Gov Expenditure"]
        BOND["Gov Dom Bonds"]
        NFA_GOV["NFA_gov (Reserves)"]
        PIF_NODE["PIF (Sovereign Wealth)"]
        GOV_REV -->|"-  surplus reduces"| BOND
        GOV_EXP -->|"+ deficit raises"| BOND
        GOV_REV -->|"+ surplus to PIF"| PIF_NODE
        NFA_GOV -->|"+ returns extracted"| GOV_REV
        PIF_NODE -->|"+ accumulates"| NFA_GOV
    end

    subgraph "Supply Side (Section 6-7)"
        Qs["Supply Choice Q_s"]
        Qd["Aggregate Demand Q_d"]
        LEONTIEF["Leontief Production fn"]
        Ys["Realized Supply Y_s"]
        Qs -->|"+"| LEONTIEF
        Qd -->|"+"| Ys
        LEONTIEF -->|"+ constrained"| Ys
    end

    subgraph "IO Structure (Section 10)"
        A_MATRIX["A Matrix (IO Table)"]
        X["Total Output X_(t)"]
        V2030["Vision 2030 Policy"]
    end

    subgraph "Energy Module (Section 11)"
        ENERGY["Energy Demand"]
        EMISSIONS["Emissions"]
        RE["Renewables"]
        NETZERO["Net Zero Offsets"]
    end

    subgraph "Accounting (Section 12)"
        K["Capital Stock K"]
        PROFITS["Profits P_(t)"]
        HH["HH Accounting"]
        BANKS["Bank Profits"]
    end

    subgraph "Biophysical (Section 13)"
        WATER["Water Module"]
    end

    C -->|"+"| Qd
    GY -->|"+"| Qd
    I -->|"+"| Qd
    EX -->|"+"| Qd
    IM["Imports IM"] -->|"-"| Qd

    Ys -->|"+ Leontief"| X
    A_MATRIX -->|"+ matrix mult"| X
    V2030 -->|"+ growth targets"| A_MATRIX
    X -->|"-  unit cost share"| P_prices
    PROFITS -->|"+ markup"| P_prices

    X -->|"+ intensity"| ENERGY
    ENERGY -->|"+ combustion"| EMISSIONS
    RE -->|"-  displaces fossil"| EMISSIONS
    NETZERO -->|"-  removes"| EMISSIONS
    EMISSIONS -->|"-  carbon cost"| PROFITS

    I -->|"+"| K
    K -->|"+ capacity"| LEONTIEF
    W -->|"+ labour"| LEONTIEF

    X -->|"+ gross output"| PROFITS
    W -->|"-  wage cost"| PROFITS
    PROFITS -->|"+"| HH
    W -->|"+"| HH
    HH -->|"+"| WEALTH_lag
    I -->|"+ loans"| BANKS
    BANKS -->|"+ returns"| HH

    Ys -->|"+ gap fills"| IM
    X -->|"+ demand"| WATER
    WATER -->|"+ signal"| I_desal

    REMIT -->|"-  outflow"| NFA_GOV
    GOV_REV -->|"+"| GOV_EXP
    IM -->|"-  trade deficit"| NFA_GOV
    PROFITS -->|"+ tax base"| GOV_REV
    EX -->|"+ trade surplus"| NFA_GOV
    C -->|"+ VAT"| GOV_REV
    GY -->|"+"| GOV_EXP
```

## 2. Prices, Wages & Consumption
model.py §3.1–3.4: price formation (EMA normal-cost pricing), inflation, wage determination, household consumption.

```mermaid
graph LR
    subgraph "Price Formation & Inflation"
        X_lag["X_(t-1) Total Output"] -->|"-  unit cost share"| UCOST["Unit Cost EMA unit_cost_smooth_"]
        P_lag["Profits P_(t-1)"] -->|"-  reduces cost share"| UCOST
        UCOST -->|"+ normal-cost markup theta"| P_price["Prices p_(t)"]
        INF_E["Expected Inflation"] -->|"+ cost push"| P_price
        P_price -->|"+ Laspeyres index"| INF["GDP Deflator"]
        P_price -->|"+ consumer weights dC"| INF_C["Consumer Price Index"]
        INF -->|"+ persistence"| INF_E
        P_price -->|"fixed at 1"| DESAL_P["Desal/Wwater Price = 1"]
    end

    subgraph "Wage Determination"
        INF_lag["Inflation(t-1)"] -->|"+ indexation"| gW["Wage growth gW"]
        POP["Population growth adj."] -->|"+ additive"| gW
        gW -->|"+"| W["Wages W_(t)"]
        X_lag2["X_(t-1)"] -->|"+ wage share"| W
        SAUD["Saudization rate"] -->|"-  reduces foreign wage bill"| W
    end

    subgraph "Household Consumption"
        YD_W["Disposable Wage Income YD_wage"] -->|"+ MPC alpha1"| C["Consumption C(t)"]
        YD_P["Disposable Profit Income YD_profit"] -->|"+ MPC alpha2"| C
        V_lag["HH Wealth V(t-1)"] -->|"+ MPC alpha3"| C
        INF_E2["Inflation Forecast"] -->|"+ nominal adj"| C
        TAU_VAT["VAT rate tau_vat"] -->|"-  reduces net"| C
        REMIT["Remittances(t)"] -->|"-  reduces YD_wage"| YD_W
        C -->|"+ composition dC"| C_sect["Sectoral Consumption C_(t)"]
    end

    W -->|"+"| YD_W
    TAX_W["Wage Tax"] -->|"-"| YD_W
    TH["HH Taxes TH"] -->|"-"| YD_P
    P_lag -->|"+"| YD_P
    FDI_INC["FDI Income Payments"] -->|"-  repatriation"| YD_P
    PB_HH["Bank Profits to HH"] -->|"+"| YD_P
```

## 3. Investment Module
investment_module.py + energy_investment_module.py + FDI_module.py

```mermaid
graph TB
    subgraph "Energy Investment Module"
        ELEC_LOAD["Electrification Added Load"] -->|"+ demand"| RE_TARGET["RE Generation Target"]
        V2030_RE["Vision 2030 RE Ramp"] -->|"+ policy"| RE_TARGET
        NZ_RE["Net Zero 85pct Target"] -->|"+ policy"| RE_TARGET
        RE_TARGET -->|"+ gap"| NEW_CAP["New RE Capacity Needed"]
        K_RE_lag["RE Capital(t-1)"] -->|"-  depreciated existing"| NEW_CAP
        NEW_CAP -->|"+ solar share"| I_SOLAR["I_solar"]
        NEW_CAP -->|"+ wind share"| I_WIND["I_wind"]
        NEW_CAP -->|"+ bio share"| I_BIO["I_bio"]
        I_SOLAR -->|"+"| I_RE["Total RE Investment I_RE"]
        I_WIND -->|"+"| I_RE
        I_BIO -->|"+"| I_RE
    end

    subgraph "Hydrogen Investment"
        P_H2["H2 capacity growth"] -->|"+"| I_H2["I_hydrogen"]
        K_H2["H2 capital stock"] -->|"+ capacity"| H2_PROD["H2 Production"]
    end

    subgraph "Net Zero Investment"
        CCS["CCS capacity ramp"] -->|"+"| I_CCS["I_netzero_ccs"]
        NAT["Nature-based SGI"] -->|"+"| I_NAT["I_netzero_nature"]
        CCU["CCU growth"] -->|"+"| I_CCU["I_netzero_ccu"]
    end

    subgraph "Efficiency & Electrification"
        EFF_GAIN["Efficiency annual gains"] -->|"+"| I_EFF["I_efficiency_total"]
        FOSSIL["Fossil demand"] -->|"+ annual rate shift"| I_ELECTR["I_electrification_total"]
        I_ELECTR -->|"+ adds load"| ELEC_LOAD
    end

    subgraph "Storage"
        RE_VAR["RE variability"] -->|"+ need"| I_STOR["I_storage"]
        LOCAL_SHARE["Local content ramp"] -->|"+"| I_STOR_LOCAL["I_storage_local"]
        I_STOR -->|"+"| I_STOR_LOCAL
    end

    subgraph "Desalination & Wastewater"
        WATER_GAP["Water gap"] -->|"+"| I_DESAL["I_desal"]
        V2030_WATER["Vision 2030 high growth"] -->|"+"| I_DESAL
        I_DESAL -->|"+"| K_DESAL["Desal Capital"]
        WATER_GAP -->|"+"| I_WWATER["I_wastewater"]
    end

    subgraph "General Sectoral Investment"
        I_TREND["gI_trend_endog"] -->|"+"| I_SECT["Sectoral I_(t)"]
        Y_DIST["Y_(t-1) sectoral shares"] -->|"+"| I_SECT
        EXCESS_IMP["Excess imports"] -->|"+ scarcity signal"| I_SECT
        K_GAP["Capital scarcity"] -->|"+ scarcity signal"| I_SECT
        I_RE -->|"+"| I_SECT
        I_H2 -->|"+"| I_SECT
        I_CCS -->|"+"| I_SECT
        I_NAT -->|"+"| I_SECT
        I_CCU -->|"+"| I_SECT
        I_EFF -->|"+"| I_SECT
        I_ELECTR -->|"+"| I_SECT
        I_STOR -->|"+"| I_SECT
    end

    subgraph "FDI Decomposition"
        I_SECT -->|"+ base"| FDI["FDI net flows"]
        I_SECT -->|"+ total"| I_DOM["Domestic Investment I_dom"]
        FDI -->|"-  substitutes"| I_DOM
        I_DOM -->|"+ private share"| I_PRIV_DOM["I_private_dom"]
        I_DOM -->|"+ public share"| I_PUB_DOM["I_public_dom"]
    end

    subgraph "Demand Vectors"
        I_SECT -->|"+ dI, dI_RE, dI_Effic vectors"| I_DEMAND["I_demand_(t)"]
        GRID["Grid upgrade demand"] -->|"+"| I_DEMAND
    end
```

## 4. Exports, Imports, Green Exports & Balance of Payments
exports_module.py + green_exports_module.py + FDI_module.py BoP closure.

```mermaid
graph TB
    subgraph "Oil Exports"
        OIL_TS["Oil Activity Time Series"] -->|"+ AR model"| OIL_FORECAST["Oil Growth Forecast"]
        OIL_ADJ["Oil growth adjustment param"] -->|"+"| OIL_G["Effective Oil Growth"]
        OIL_FORECAST -->|"+"| OIL_G
        SHOCK["Random Oil Shock sigma"] -->|"+/- stochastic"| OIL_G
        OIL_G -->|"+"| EX_OIL["Oil Exports EX_oil_(t)"]
        EX_OIL_lag["EX_oil_(t-1)"] -->|"+"| EX_OIL
        X_lag["X_(t-1)"] -->|"-  supply ceiling"| EX_OIL
    end

    subgraph "Non-Oil Exports"
        NONOIL_TS["Non-Oil Export Time Series"] -->|"+ AR model"| NONOIL_FORECAST["Non-Oil Growth Forecast"]
        NONOIL_ADJ["Non-oil export growth adj."] -->|"+"| NONOIL_G["Effective Non-Oil Growth"]
        V2030_EX["V2030 sectoral growth rates"] -->|"+ scaling factor"| NONOIL_G
        NONOIL_FORECAST -->|"+"| NONOIL_G
        NONOIL_G -->|"+"| EX_NONOIL["Non-Oil Exports EX_non_oil_(t)"]
        EX_NONOIL_lag["EX_non_oil_(t-1)"] -->|"+"| EX_NONOIL
    end

    subgraph "Green Exports  -  green_exports_module.py"
        H2_EXPORT["H2 Export Volume"] -->|"+ price per GJ"| H2_EX_VAL["H2 Export Value"]
        H2_EX_VAL -->|"+"| EX_NONOIL
        SOLAR_INV["Solar/Battery Investment"] -->|"+ local content pct"| SOLAR_EX["Solar Export Value"]
        SOLAR_EX -->|"+"| EX_NONOIL
        SOLAR_INV -->|"-  import substitution"| IM_REDUCE["Reduced Imports IM_(t)"]
        WIND_INV["Wind/Electrification Inv"] -->|"+ local content pct"| WIND_EX["Wind/Mach Export"]
        WIND_EX -->|"+"| EX_NONOIL
        WIND_INV -->|"-  import substitution"| IM_REDUCE
        CARBON_CREDITS["Carbon Credit Export"] -->|"+"| EX_NONOIL
        STORAGE_LOCAL["Local Storage Production"] -->|"-"| IM_REDUCE
    end

    subgraph "Imports"
        IM_lag["IM_(t-1)"] -->|"+"| IM["Imports IM_(t)"]
        IM_ADJ["Import growth adj."] -->|"+"| IM
        UNMET["Unmet Demand"] -->|"+ residual fill"| IM
        IM_REDUCE -->|"-"| IM
    end

    subgraph "Balance of Payments (BPM6)"
        EX_OIL -->|"+"| EX_TOTAL["Total Exports EX_(t)"]
        EX_NONOIL -->|"+"| EX_TOTAL
        EX_TOTAL -->|"+"| TRADE_BAL["Trade Balance"]
        IM -->|"-"| TRADE_BAL
        NFA_GOV_LAG["NFA_gov(t-1)"] -->|"+ r_reserve"| NFA_INC["NFA Income"]
        NFA_INC -->|"+ extraction rate e"| PRIM_INC["Primary Income Balance"]
        FDI_STOCK_LAG["FDI_net_stock(t-1)"] -->|"+ r_FDI"| FDI_INC["FDI Income Payments"]
        FDI_INC -->|"-  repatriation"| PRIM_INC
        BOND_EXT_INT["Ext Bond Interest"] -->|"-"| PRIM_INC
        REMIT["Remittances"] -->|"-"| SEC_INC["Secondary Income Balance"]
        TRADE_BAL -->|"+"| CURR_ACC["Current Account"]
        PRIM_INC -->|"+"| CURR_ACC
        SEC_INC -->|"+"| CURR_ACC
        CURR_ACC -->|"+"| DELTA_NFA["delta NFA_gov"]
        NFA_EXTRACT["NFA Income Reinvested (1-e)"] -->|"+"| DELTA_NFA
        GOV_PIF["Gov PIF Outflow"] -->|"-"| DELTA_NFA
        BOND_EXT_ISS["Ext Bond Issuance"] -->|"+"| DELTA_NFA
        FDI_NET["FDI Net Inflow"] -->|"+"| DELTA_NFA
        NFA_GOV_LAG -->|"+ carry"| NFA_GOV
        DELTA_NFA -->|"+"| NFA_GOV["NFA_gov(t)"]
    end
```

## 5. Supply-Demand Reconciliation & Leontief Production
model.py §6–10: expectations, aggregate demand, Leontief production function, unmet demand.

```mermaid
graph TB
    subgraph "Supply Decision  -  Section 6"
        gY_e["Expected Demand Growth gY_e"] -->|"+"| Qs["Supply Choice Q_s_(t)"]
        Y_lag["Y_(t-1) Realized GDP"] -->|"+"| Qs
        Y_TS["Y Time Series AR Model"] -->|"+ forecast"| gY_e
        GOV_BOOST["Gov Confidence Boost"] -->|"+"| Qs
        Qs -->|"+ nominal to real"| qs["Real Supply q_s"]
    end

    subgraph "Aggregate Demand  -  Section 6.2"
        C["Consumption C"] -->|"+"| Qd["Demand Q_d_(t)"]
        GY["Gov Spending GY"] -->|"+"| Qd
        I_DEM["Investment Demand I_demand"] -->|"+"| Qd
        EX_OIL["Oil Exports"] -->|"+"| Qd
        EX_NONOIL["Non-Oil Exports"] -->|"+"| Qd
        IM["Imports (subtract)"] -->|"-"| Qd
    end

    subgraph "Leontief Production  -  Section 7"
        BETA["beta * IntP(t-1)"] -->|"+ plus flex"| MIN1["Min: IntP capacity vs K capacity"]
        KAPPA["kappa * K(t-1)"] -->|"+ times flex_capital"| MIN1
        ALPHA["alpha * W(t)"] -->|"+"| MIN2["Min: Labor capacity vs Min1"]
        MIN1 -->|"+"| MIN2
        Qs -->|"+"| Ys["Realized Supply Y_s_(t)"]
        MIN2 -->|"-  binding min"| Ys
    end

    subgraph "Binding Constraints  -  Section 7.1"
        Qs -->|"-  gap"| GAP_INT["IntP Binding Gap"]
        BETA -->|"+ capacity"| GAP_INT
        Qs -->|"-  gap"| GAP_K["Capital Binding Gap"]
        KAPPA -->|"+ capacity"| GAP_K
        Qs -->|"-  gap"| GAP_W["Wage Binding Gap"]
        ALPHA -->|"+ capacity"| GAP_W
    end

    subgraph "Unmet Demand and Reconciliation  -  Section 8"
        Qd -->|"+"| UNMET["Unmet Demand = Q_d - Y_s"]
        Ys -->|"-"| UNMET
        UNMET -->|"+ option 1"| IM_ADJ["Increase Imports"]
        UNMET -->|"-  option 2"| REDUCE_EX["Reduce Non-Oil Exports"]
        UNMET -->|"-  option 2"| REDUCE_GY["Reduce Gov Consumption"]
        UNMET -->|"-  option 2"| REDUCE_C["Reduce HH Consumption"]
        UNMET -->|"-  option 2"| REDUCE_I["Reduce Investment"]
    end

    subgraph "IO Structure and Output  -  Section 9-10"
        Ys -->|"+"| Y_FINAL["Realized Y_(t)"]
        Y_FINAL -->|"+ Leontief"| X["Total Output X = inv(I-A) * Y"]
        A["A Matrix"] -->|"+"| X
        V2030["Vision 2030 Policy"] -->|"+ direct sector growth"| Y_FINAL
        V2030 -->|"+ cluster supply chain growth"| Y_FINAL
        V2030 -->|"-  import substitution"| IM
    end

    GAP_INT -->|"+ scarcity signal"| I_ENDOG["Endogenous Investment Growth"]
    GAP_K -->|"+ scarcity signal"| I_ENDOG
```

## 6. Energy Module
energy_module.py: energy demand, generation mix, emissions, net zero, carbon market.

```mermaid
graph TB
    subgraph "Energy Demand  -  Section 1-2"
        X["Total Output X_(t)"] -->|"+ gas intensity"| GAS["Gas Use_(t)"]
        X -->|"+ oil intensity"| OIL["Oil Use_(t)"]
        X -->|"+ elec intensity"| ELEC["Electricity Use_(t)"]
        EFF["Energy Reduction Factor"] -->|"-  decreases demand"| GAS
        EFF -->|"-  decreases demand"| OIL
        EFF -->|"-  decreases demand"| ELEC
        OIL_RED["Oil-to-Gas Switching V2030"] -->|"-  reduces oil"| OIL
        OIL_RED -->|"+ adds to gas"| GAS
    end

    subgraph "Electrification Strategy"
        ELECTR_RATE["Annual Electrification Rate"] -->|"+ cumulative shift"| FOSSIL_SHIFT["Fossil to Electricity Shift"]
        FOSSIL_SHIFT -->|"-  replaces"| OIL
        FOSSIL_SHIFT -->|"-  replaces"| GAS
        FOSSIL_SHIFT -->|"+ adds load"| ELEC
    end

    subgraph "Hydrogen Module"
        K_H2["H2 Physical Capital"] -->|"+ capacity"| H2_PROD["H2 Production"]
        H2_PROD -->|"+ efficiency factor"| RE_USED_H2["RE Used for H2"]
        H2_PROD -->|"+ export share"| H2_EXPORT["H2 Export"]
        H2_PROD -->|"+ domestic share"| H2_DOM["H2 Domestic Use"]
        H2_DOM -->|"-  substitutes"| GAS
    end

    subgraph "Electricity Generation Mix  -  Section 3"
        ELEC -->|"+ total demand"| TOTAL_ELEC["Total Elec Generation"]
        RE_GEN["RE Generation Capacity"] -->|"+ net of H2 use"| RE_ELEC["RE for Electricity"]
        RE_ELEC -->|"-  displaces fossil"| FF_GAP["Fossil Fuel Gap"]
        FF_GAP -->|"+ oil share declining"| OIL_ELEC["Oil for Electricity"]
        FF_GAP -->|"+ gas share rising"| GAS_ELEC["Gas for Electricity"]
    end

    subgraph "Fossil Substitution by Surplus RE  -  Section 4"
        RE_SURPLUS["Surplus RE after Elec"] -->|"-  substitutes"| OIL_NONEL["Non-Elec Oil Use"]
        RE_SURPLUS -->|"-  substitutes"| GAS_NONEL["Non-Elec Gas Use"]
    end

    subgraph "Emissions  -  Section 5"
        OIL -->|"+ emission factor"| OIL_EMIS["Oil Emissions"]
        GAS -->|"+ emission factor"| GAS_EMIS["Gas Emissions"]
        ELEC -->|"+ dynamic factor"| ELEC_EMIS["Electricity Emissions"]
        OIL_EMIS -->|"+"| GROSS_EMIS["Gross Emissions"]
        GAS_EMIS -->|"+"| GROSS_EMIS
        ELEC_EMIS -->|"+"| GROSS_EMIS
    end

    subgraph "Net Zero Offsets"
        CCU["CCU Capacity"] -->|"+"| REUSE["Flow: Reuse/Recycle"]
        CCS["CCS Ramp 2035 target"] -->|"+"| REMOVE_CCS["Flow: Remove CCS"]
        SGI["Saudi Green Initiative"] -->|"+"| REMOVE_NAT["Flow: Remove Nature"]
        REUSE -->|"+"| PHYS_REMOVAL["Physical Removal"]
        REMOVE_CCS -->|"+"| PHYS_REMOVAL
        REMOVE_NAT -->|"+"| PHYS_REMOVAL
        GROSS_EMIS -->|"+"| NET_EMIS["Net Emissions"]
        PHYS_REMOVAL -->|"-  subtracts"| NET_EMIS
    end

    subgraph "Carbon Market"
        NET_EMIS -->|"+"| DEMAND_CC["Carbon Credit Demand"]
        REMOVE_CCS -->|"+"| SUPPLY_CC["Carbon Credit Supply"]
        REMOVE_NAT -->|"+"| SUPPLY_CC
        DEMAND_CC -->|"+"| TRADED["Traded Volume"]
        SUPPLY_CC -->|"+"| TRADED
        TRADED -->|"-  cost to firms"| CC_COST["Carbon Credit Cost to P_(t)"]
        TRADED -->|"+ revenue to producers"| CC_REV["Carbon Credit Revenue to P_(t)"]
    end
```

## 7. Stock-Flow Accounting
model.py §12: GDP, capital dynamics, profits, government, banks, households, sovereign wealth.

```mermaid
graph TB
    subgraph "12.1 GDP and Growth"
        Y["Y_(t) = C + GY + I_demand + EX - IM"]
        Y -->|"+ deflate"| y["Real GDP y_(t)"]
        Y -->|"+ oil sectors mask"| Y_OIL["Oil GDP"]
        Y -->|"+ non-oil mask"| Y_NONOIL["Non-Oil GDP"]
        Y -->|"+ pct change"| gY["GDP Growth Rate gY"]
        X["Total Output X"] -->|"+"| dX["Output Shares dX_(t)"]
    end

    subgraph "12.2 Capital Dynamics"
        K_lag["K_(t-1)"] -->|"-  depreciate delta"| K["K_(t) = K(t-1)(1-delta) + I(t-1)"]
        I_lag["I_(t-1)"] -->|"+"| K
        K -->|"+ utilization"| u["Capacity Utilization u"]
        x["Real output x"] -->|"+"| u
    end

    subgraph "12.3 Profits"
        X2["X_(t)"] -->|"+"| P["Profits P_(t)"]
        IntP["Intermediate Purchases"] -->|"-"| P
        W["Wages W_(t)"] -->|"-"| P
        TAX_PROD["Production Taxes"] -->|"-"| P
        TAX_PROD_NET["Product Taxes net"] -->|"-"| P
        SUB["Production Subsidies"] -->|"+"| P
        TAX_INV["Investment Taxes"] -->|"-"| P
        CC_COST["Carbon Credit Costs"] -->|"-"| P
        CC_REV["Carbon Credit Revenue"] -->|"+"| P
    end

    subgraph "12.4 Government"
        VAT["VAT = C * tau_vat"] -->|"+"| GOV_REV["Gov Revenue"]
        TH["HH Taxes on profits"] -->|"+"| GOV_REV
        GP["Public Profits"] -->|"+"| GOV_REV
        TAX_w["Wage Tax"] -->|"+"| GOV_REV
        TAX_p["Profit Tax"] -->|"+"| GOV_REV
        TAX_PROD2["Production/Product Taxes"] -->|"+"| GOV_REV
        NFA_EXTRACT["NFA Income (extracted share e)"] -->|"+"| GOV_REV
        GY["Gov Consumption"] -->|"+"| GOV_EXP["Gov Expenditure"]
        SUB2["Subsidies"] -->|"+"| GOV_EXP
        GI["Gov Investment"] -->|"+"| GOV_EXP
        BOND_INT["Dom Bond Interest+Repayment"] -->|"+"| GOV_EXP
        EXT_BOND_SRV["Ext Bond Service"] -->|"+"| GOV_EXP
        GOV_EXP -->|"+ deficit"| BOND["Dom Bonds B(t)"]
        GOV_REV -->|"-  surplus"| BOND
        GOV_REV -->|"+"| NET_W["Gov Net Wealth"]
        GOV_EXP -->|"-"| NET_W
    end

    subgraph "12.5 Firms and Banks (Endogenous Money)"
        I_PRIV["Private Investment"] -->|"+ loan share"| L["Loans L_(t)"]
        L_lag["L_(t-1)"] -->|"-  repayment rho"| L
        L -->|"= endogenous money"| D_HH["HH Deposits D_HH = L"]
        L -->|"+ interest rl"| PB["Bank Profits PB"]
        BOND3["Gov Dom Bonds"] -->|"+ interest rm"| PB
        V2["HH Wealth V"] -->|"+"| PB
    end

    subgraph "12.6 Household Accounting"
        W2["Wages"] -->|"+"| YD_W["Disposable Wage Income YD_wage"]
        TAX_W2["Wage Tax"] -->|"-"| YD_W
        REMIT["Remittances"] -->|"-  outflow"| YD_W
        P2["Private Profits distributed"] -->|"+"| YD_P["Disposable Profit Income YD_profit"]
        PB2["Bank Profits to HH"] -->|"+"| YD_P
        TH2["HH Taxes TH"] -->|"-"| YD_P
        LOAN_INT["Loan Interest"] -->|"-"| YD_P
        FDI_INC["FDI Income Payments"] -->|"-  repatriation"| YD_P
        YD_W -->|"+"| V["HH Wealth V(t)"]
        YD_P -->|"+"| V
        C2["Consumption"] -->|"-"| V
        VAT2["VAT"] -->|"-"| V
        I_PROF["Investment from profits"] -->|"-"| V
        REMIT2["Remittances"] -->|"-"| V
        D_HH2["HH Deposits D_HH"] -->|"+ asset"| HH_NW["HH Total Net Worth"]
        V -->|"+ portfolio claims"| HH_NW
        E_FIRM_HH["Firm Equity HH share E_firm_HH"] -->|"+"| HH_NW
        E_BANK_HH["Bank Equity HH share E_bank_HH"] -->|"+"| HH_NW
    end

    subgraph "12.7 Sovereign Wealth & External Sector"
        NFA_GOV_LAG["NFA_gov(t-1)"] -->|"+ r_reserve"| NFA_INC["NFA Income"]
        NFA_INC -->|"+ extraction e"| NFA_EXTRACT
        NFA_INC -->|"- reinvested (1-e)"| NFA_REINV["NFA Reinvested"]
        NFA_REINV -->|"+"| NFA_GOV["NFA_gov(t)"]
        NFA_GOV_LAG -->|"+ carry"| NFA_GOV
        PIF_OUT -->|"-"| NFA_GOV
        GOV_SURPLUS["Gov Fiscal Surplus"] -->|"+ pif_rate"| PIF["PIF Asset Stock"]
        GOV_REV2["Gov Revenue"] -->|"+"| GOV_SURPLUS
        GOV_EXP2["Gov Expenditure"] -->|"-"| GOV_SURPLUS
        PIF -->|"+ foreign invest"| PIF_OUT["PIF Outflow"]
        NFA_GOV -->|"-  below floor"| BOND_EXT_TRIG["Ext Bond Trigger"]
        BOND_EXT_TRIG -->|"+"| BOND_EXT["Ext Bond Stock B_ext"]
        BOND_EXT -->|"+ debt service"| EXT_BOND_SRV
        OFA_HH["OFA_HH"] -->|"+ TFM residual"| OFA_SUM["Sum OFA = 0 (closure)"]
        OFA_GOV["OFA_Gov"] -->|"+"| OFA_SUM
        OFA_FIRMS["OFA_Firms"] -->|"+"| OFA_SUM
        OFA_BANKS["OFA_Banks"] -->|"+"| OFA_SUM
        OFA_ROW["OFA_ROW"] -->|"+"| OFA_SUM
    end
```

## 8. IO Table & Endogenous A Matrix
calc_io_table (model.py) + endogenize_input_output_matrix_A.py

```mermaid
graph TB
    subgraph "IO Table Dynamics  -  calc_io_table"
        K_DESAL["Desal Capital K_desal(t-1)"] -->|"+"| DESAL_OUT["Desal Output = K * u * eK"]
        X_WATER["Water Sector Output x_water(t-1)"] -->|"-  normalise"| DESAL_SHARE["Desal Share = Desal_Out / x_water"]
        DESAL_OUT -->|"+"| DESAL_SHARE
        DESAL_SHARE -->|"-  reduces other inputs"| A_WATER["A col water * (1 - desal_share)"]
        DESAL_SHARE -->|"+ adds desal row"| A_DESAL["A desal water = desal_share"]
        A_WATER -->|"+"| A_NEW["Updated A Matrix"]
        A_DESAL -->|"+"| A_NEW
    end

    subgraph "Endogenize IO Matrix A  -  Vision 2030"
        EXCESS_IMP["Excess Import Measure"] -->|"+"| RANK["Sector Import Dependency Rank"]
        SUPPLY_SCARCITY["Supply Constraint Metrics"] -->|"+"| RANK

        subgraph "Step 1: Direct V2030 Sector Growth"
            V2030_RATES["V2030 Sectoral Growth Rates"] -->|"+"| Y_INCREASE["Increase Y_(t) for V2030 sectors"]
            Y_INCREASE -->|"-  substitution"| IM_DECREASE["Decrease Imports of V2030 sectors"]
            Y_INCREASE -->|"+"| gI_INCREASE["Increase Endogenous Investment Growth"]
        end

        subgraph "Step 2: Cluster Supply Chain Growth"
            V2030_RATES2["V2030 Growth Rates"] -->|"+"| INTP_PROXY["Increase IntP for V2030 columns"]
            INTP_PROXY -->|"+"| SUPPLIER_INCREASE["Supplier Sector Y_(t) Increase"]
            SUPPLIER_INCREASE -->|"-  substitution"| IM_SUPPLIER["Decrease Supplier Imports"]
            SUPPLIER_INCREASE -->|"+"| gI_SUPPLIER["Increase Supplier Investment Growth"]
        end
    end

    subgraph "Output Calculation"
        A_NEW -->|"+ Leontief inverse"| X_CALC["X = inv(I - A) * Y  Leontief inverse"]
        Y_FINAL["Realized Y_(t)"] -->|"+"| X_CALC
        X_CALC -->|"+"| Z["Z Matrix = A * diag(X)"]
        X_CALC -->|"+"| INTP["IntP = sum(A,0) * X"]
        X_CALC -->|"+"| INTS["IntS = A * X"]
    end
```

## 9. Water Module
water_module.py: crop demand, irrigation, water balance, cost accounting.

```mermaid
graph TB
    subgraph "Water Module"
        subgraph "Crop Demand Model"
            POP["Population SSP growth"] -->|"+"| FOOD_DEM["Per Capita Food Demand"]
            SSR["Self-Sufficiency Ratios scenario"] -->|"+"| DOM_PROD["Domestic Production Required"]
            FOOD_DEM -->|"+"| DOM_PROD
            YIELD["Crop Yield ton per m2"] -->|"-  fewer area needed"| AREA["Cultivated Area"]
            DOM_PROD -->|"+"| AREA
        end

        subgraph "Irrigation Technology"
            AREA -->|"+"| IRR["Irrigation Water Demand"]
            BWF["Blue Water Footprint coeff."] -->|"+"| IRR
            TECH_MIX["Technology Mix Surface/Sprinkler/Drip"] -->|"+"| EFF["Irrigation Efficiency"]
            EFF -->|"-  reduces demand"| IRR
            GH_PCT["Greenhouse Production pct"] -->|"-  reduces field area"| IRR
        end

        subgraph "Water Supply"
            K_DESAL["Desal Capital Stock"] -->|"+"| POT_DESAL["Desalination Potential"]
            K_WWATER["Wastewater Capital"] -->|"+"| POT_WWATER["Wastewater Potential"]
            POT_DESAL -->|"+"| TOTAL_SUPPLY["Total Water Supply"]
            POT_WWATER -->|"+"| TOTAL_SUPPLY
            GROUND["Groundwater residual"] -->|"+"| TOTAL_SUPPLY
        end

        subgraph "Water Balance"
            IRR -->|"+"| WATER_USE["Total Water Use National"]
            TOTAL_SUPPLY -->|"-  closes gap"| GAP["Water Supply-Demand Gap"]
            WATER_USE -->|"+"| GAP
            GAP -->|"+ if gap positive"| I_DESAL_SIGNAL["Signal: Increase Desal Investment"]
            GAP -->|"+ if gap positive"| I_WWATER_SIGNAL["Signal: Increase Wwater Investment"]
        end

        subgraph "Cost Accounting"
            AREA2["Irrigated Area"] -->|"+"| INVEST_COST["Irrigation Investment Cost"]
            TECH_MIX2["Technology Mix"] -->|"+"| INVEST_COST
            AREA2 -->|"+"| OM_COST["O and M Cost"]
            SUBSIDY_RATE["Subsidy Rate scenario"] -->|"-  reduces"| NET_COST["Net Cost after Subsidy"]
            INVEST_COST -->|"+"| NET_COST
            OM_COST -->|"+"| NET_COST
        end

        subgraph "Nature-Energy-Water Nexus"
            SGI["Saudi Green Initiative trees"] -->|"+ 0.315 Gt water per Mt CO2"| SGI_WATER["SGI Water Demand"]
            SGI_WATER -->|"+"| WATER_USE
        end
    end
```

## 10. Government Sector
model.py §5.2 & §12.4: revenue, expenditure, bonds, countercyclical policy.

```mermaid
graph TB
    subgraph "Government Sector"
        subgraph "Government Revenue"
            C["Consumption C"] -->|tau_vat| VAT["VAT"]
            P_PRIV["Private Profits"] -->|tau_HH| TH["HH Taxes"]
            P_PUB["Public Profits oil rents"] --> GP["Gov Oil Profits"]
            W["Wages"] -->|wage tax rate| TAX_W["Wage Tax"]
            P_PRIV2["Private Profits"] -->|profit tax rate| TAX_P["Profit Tax"]
            X["Output X"] -->|tau_production| TAX_PROD["Production Tax"]
            X -->|tau_products| TAX_PROD_NET["Product Tax net"]
            VAT --> GOV_REV["Total Gov Revenue"]
            TH --> GOV_REV
            GP --> GOV_REV
            TAX_W --> GOV_REV
            TAX_P --> GOV_REV
            TAX_PROD --> GOV_REV
            TAX_PROD_NET --> GOV_REV
        end

        subgraph "Government Expenditure"
            GY_TS["GY Time Series AR Model"] -->|or avg growth| GY["Gov Consumption GY"]
            OIL_SHOCK["Oil Export Shock lag 1"] -->|cyclicality factor| GY
            GOV_BOOST["Confidence Boost"] --> GY
            GY -->|sectoral dG| GY_SECT["Sectoral GY_(t)"]
            X2["Output X"] -->|subsidy rate| SUB["Production Subsidies"]
            GI["Public Investment"] --> GOV_EXP["Total Gov Expenditure"]
            GY --> GOV_EXP
            SUB --> GOV_EXP
            BOND_INT["Bond Interest plus Repayment"] --> GOV_EXP
            RESID["Gov Expenditure Residual"] --> GOV_EXP
        end

        subgraph "Government Balance Sheet"
            GOV_EXP --> BOND["Gov Bonds B(t)"]
            GOV_REV --> BOND
            B_lag["Bonds B(t-1)"] -->|repayment rho| BOND
            GOV_REV --> NET_W["Gov Net Wealth"]
            GOV_EXP --> NET_W
            ARAMCO["Aramco Market Cap"] --> NET_W_PLUS["Gov Net Wealth plus Aramco"]
            NET_W --> NET_W_PLUS
        end

        subgraph "Countercyclical Policy"
            gY_neg["Negative GDP growth"] --> BOOST["Gov Confidence Boost"]
            INF_neg["Negative Inflation"] --> BOOST
            BOOST -->|factor| I_PUB_BOOST["Boost Public Investment"]
            BOOST --> GY
            BOOST --> Qs["Supply Choice Q_s"]
        end
    end
```

## 11. FDI Module
FDI_module.py: foreign direct investment decomposition and balance of payments.

```mermaid
graph TB
    subgraph "FDI Module"
        subgraph "FDI Growth Modes"
            MODE_A["Mode A: Follow Domestic Growth"] --> gI_REAL["Sectoral I growth = I(t)/I(t-1) - 1"]
            gI_REAL --> FDI_SECT["Sectoral FDI Growth"]
            FDI_TARGET["FDI Target V2030 window"] -->|premium| FDI_SECT
            FDI_SECT -->|cap 50 pct per sector| FDI_NET["FDI Net Flow_(t)"]

            MODE_B["Mode B: Fixed Growth Rate"] --> FDI_FIXED["FDI = Base * (1+target)^t"]
            FDI_FIXED -->|V2030 timing cap| FDI_NET
        end

        subgraph "FDI Accounting"
            FDI_NET -->|distributed by I shares| FDI_FLOW["FDI_net_flow_(t)"]
            FDI_FLOW --> FDI_GROSS["FDI Gross Flow"]
            FDI_FLOW --> FDI_STOCK["FDI Stock cumulative"]
            FDI_FLOW --> SHARE["FDI Share of Investment"]
        end

        subgraph "Impact on Domestic Financing"
            I_TOTAL["Total Investment I_(t)"] --> I_DOM["I_domestic = I - FDI_flow"]
            FDI_FLOW --> I_DOM
            I_DOM -->|private share| I_PRIV_DOM["I_private_dom needs loans"]
            I_DOM -->|public share| I_PUB_DOM["I_public_dom gov budget"]
        end

        subgraph "Balance of Payments"
            FDI_NET --> CAP_ACC["Capital Account"]
            EX["Exports"] --> CURR_ACC["Current Account = EX - IM"]
            IM["Imports"] --> CURR_ACC
        end
    end
```

## 12. Key Feedback Loops (Reinforcing & Balancing)
Cross-cutting loops across all modules. **R** = reinforcing (positive), **B** = balancing (negative).

```mermaid
graph TB
    subgraph "R1: Multiplier-Accelerator Loop (positive)"
        Y1["GDP Y UP"] --> C1["Consumption C UP"]
        C1 --> Qd1["Demand Q_d UP"]
        Qd1 --> Y1
        Y1 --> I1["Investment I UP"]
        I1 --> Qd1
    end
```

```mermaid
graph TB
    subgraph "R2: Profit-Investment Loop (positive)"
        P1["Profits P UP"] --> YD1["HH Profit Income UP"]
        YD1 --> C2["Consumption UP"]
        C2 --> X1["Output X UP"]
        X1 --> P1
        P1 --> I2["Investment from profits UP"]
        I2 --> K1["Capital K UP"]
        K1 --> Y_S["Supply Capacity UP"]
    end
```

```mermaid
graph TB
    subgraph "R3: Wage-Price Spiral (positive)"
        INF1["Inflation UP"] --> W1["Wages UP"]
        W1 --> COST["Unit Costs UP"]
        COST --> P_PRICE["Prices UP"]
        P_PRICE --> INF1
    end
```

```mermaid
graph TB
    subgraph "B1: Supply Constraint Balancing (negative)"
        Y_GROW["GDP Growth UP"] --> Q_S["Supply Choice UP"]
        Q_S --> LEONT["Production Leontief min"]
        K_LAG["Capital K(t-1)"] --> LEONT
        LEONT -->|binding constraint| UNMET["Unmet Demand UP"]
        UNMET -->|import increase| IM1["Imports UP"]
        IM1 -->|reduces GDP| Y_DOWN["GDP DOWN"]
    end
```

```mermaid
graph TB
    subgraph "B2: Government Stabilizer (negative)"
        gY_NEG["GDP Growth below 0"] --> GOV_BOOST2["Countercyclical Boost"]
        GOV_BOOST2 --> GY2["Gov Spending UP"]
        GOV_BOOST2 --> I_PUB["Public Investment UP"]
        GY2 --> DEM["Demand UP"]
        I_PUB --> DEM
        DEM --> Y_RECOV["GDP Recovery UP"]
    end
```

```mermaid
graph TB
    subgraph "R4: Energy Transition Loop (positive)"
        I_RE["RE Investment UP"] --> RE_CAP["RE Capacity UP"]
        RE_CAP --> RE_SHARE["RE Share UP"]
        RE_SHARE --> FOSSIL_DOWN["Fossil Use DOWN"]
        FOSSIL_DOWN --> EMIS_DOWN["Emissions DOWN"]
        EMIS_DOWN --> CC_COST_DOWN["Carbon Credit Cost DOWN"]
        CC_COST_DOWN --> PROFITS_UP["Profits UP"]
        PROFITS_UP --> I_RE
    end
```

```mermaid
graph TB
    subgraph "B3: Water-Energy-Food Nexus (negative)"
        POP_GROW["Population Growth"] --> WATER_DEM["Water Demand UP"]
        SGI_TREES["SGI Tree Planting"] --> WATER_DEM
        WATER_DEM --> GAP["Water Gap UP"]
        GAP --> I_DESAL2["Desal Investment UP"]
        I_DESAL2 --> ENERGY_DEM["Energy Demand UP"]
        I_DESAL2 --> WATER_SUP["Water Supply UP"]
        WATER_SUP --> GAP
    end
```

## 13. Balance of Payments & Sovereign Wealth
Full BPM6 current account structure, NFA_gov dynamics, PIF, and two-phase sovereign financing (FDI_module.py + model.py §12.7).

```mermaid
graph TB
    subgraph "Current Account BPM6"
        EX_CA["Total Exports EX_(t)"] -->|"+"| TRADE_BAL_CA["Trade Balance"]
        IM_CA["Total Imports IM_(t)"] -->|"-"| TRADE_BAL_CA
        NFA_GOV_CA["NFA_gov(t-1)"] -->|"+  r_reserve"| NFA_INC_CA["NFA Income"]
        NFA_INC_CA -->|"+  extraction e"| PRIM_IN["Primary Income (+)"]
        FDI_STCK_CA["FDI Net Stock(t-1)"] -->|"+  r_FDI"| FDI_INC_CA["FDI Income Payments"]
        FDI_INC_CA -->|"-"| PRIM_IN
        BOND_EXT_CA["Ext Bond Stock"] -->|"+  interest"| EXT_INT_CA["Ext Bond Interest"]
        EXT_INT_CA -->|"-"| PRIM_IN
        REMIT_CA["Remittances"] -->|"-"| SEC_IN["Secondary Income"]
        TRADE_BAL_CA -->|"+"| CA["Current Account"]
        PRIM_IN -->|"+"| CA
        SEC_IN -->|"+"| CA
    end

    subgraph "Financial Account"
        FDI_FA["FDI Net Inflow"] -->|"+"| FA["Financial Account"]
        BOND_ISS_FA["Routine Ext Bond Issuance"] -->|"+"| FA
        EMRG_BONDS["Emergency Ext Bonds Phase 2"] -->|"+"| FA
        DELTA_RES["delta Reserves = -delta NFA_gov"] -->|"-"| FA
        CA -->|"+  closure"| BOP_CHECK["BoP Check = 0"]
        FA -->|"-  closure"| BOP_CHECK
    end

    subgraph "NFA_gov Dynamics"
        CA -->|"+"| DNFA["delta NFA_gov"]
        FDI_FA -->|"+"| DNFA
        BOND_ISS_FA -->|"+"| DNFA
        NFA_INC_REINV["NFA Income Reinvested (1-e)"] -->|"+"| DNFA
        PIF_OUT_NFA["PIF Outflow foreign invest"] -->|"-"| DNFA
        DNFA -->|"+"| NFA_GOV_T["NFA_gov(t)"]
        NFA_GOV_CA -->|"+  carry"| NFA_GOV_T
    end

    subgraph "PIF and Sovereign Wealth Accumulation"
        GOV_SURP_SW["Gov Fiscal Surplus"] -->|"+  gov_surplus_pif_rate"| PIF_SW["PIF Asset Stock"]
        PIF_SW -->|"+  foreign invest"| PIF_OUT_NFA
        NFA_GOV_T -->|"+  returns r_reserve"| NFA_INC_CA
        NFA_INC_CA -->|"+  extracted e"| GOV_REV_SW["Gov Revenue"]
        NFA_INC_CA -->|"+  reinvested 1-e"| NFA_INC_REINV
    end

    subgraph "Two-Phase Sovereign Financing"
        NFA_GOV_T -->|"-  floor 0.5*NFA_init"| PHASE_CHECK["NFA Floor Check"]
        PHASE_CHECK -->|"+  Phase 1: NFA above floor"| RESERVE_DRAW["Use Reserves drawdown"]
        RESERVE_DRAW -->|"-"| NFA_GOV_T
        PHASE_CHECK -->|"+  Phase 2: NFA below floor"| EMRG_ISSUE["Emergency Bond Issuance"]
        EMRG_ISSUE -->|"+"| EMRG_BONDS
        EMRG_BONDS -->|"+"| EXT_BOND_STK["Ext Bond Stock B_ext"]
        EXT_BOND_STK -->|"+  interest+repay"| EXT_DEBT_SRV2["Ext Debt Service"]
        EXT_DEBT_SRV2 -->|"+"| GOV_EXP_SW["Gov Expenditure"]
        GOV_EXP_SW -->|"-"| GOV_SURP_SW
    end
```